# FCB 1280 v8-3: U6_UT 개선 시도 (CCTV 데이터 확대 / 학습 길이·용량) + 시드 차이

- 탐지기: YOLO26s (Ultralytics 8.4.165) 고정, 1280×736
- Input: `uvg_build_dataset`(uvg_cctv), `uvg_eval3_gop60`(cctv_gop60), `uvg_B_data_prep`(bdata)
- Accelerator: GPU T4 x2, **Internet on**, 새 세션에서 위에서부터 실행 (예상 약 80분)
- **세션이 꺼지면 작업 폴더가 지워짐 → 끝나면 바로 마지막 셀(zip) 실행 후 다운로드**. 중간에 끊겨도 같은 세션이면 다시 실행 시 학습된 run은 평가만 함
- 결과: `/kaggle/working/fcb_v83_runs/` → `fcb_v83_runs.zip`
- 평가셋 5개 사용 안 함. 검증 장면과 같은 카메라(St-Marc 0~997) 데이터 사용 안 함

## 공통 (v8-2 U6_UT와 같음)
- B 어댑터 (백본 layer 2·4 출력), 출력 0 초기화
- 손실 R: 샘플별 ‖P'−P_ref‖²/‖P−P_ref‖² (P3·P4·P5) + 4 × 작은 객체 영역(P3·P4)
- 2000/1000/500/250 kbps 무작위, 좌우 반전, batch 4, AdamW 1e-3

## 비교
| run | 학습 데이터 | step | 어댑터 채널 (layer 2 / 4) | seed |
|---|---|---|---|---|
| UT_s0 | UVG 6 + Rouen 0~599 (2간격, 300장) + Sherbrooke 0~599 (2간격, 300장) = v8-2 U6_UT | 1500 | 24 / 48 (38,952 params, 1.80 GFLOPs) | 0 |
| UT_s1 | 위와 같음 | 1500 | 24 / 48 | 1 |
| UTx | UVG 6 + Rouen 0~629 (1간격, 630장) + Sherbrooke 0~1199 (2간격, 600장) | 1500 | 24 / 48 | 0 |
| UTx_lb | UTx와 같음 | 3000 | 32 / 64 (54,368 params, 2.51 GFLOPs) | 0 |

- UT_s0·UT_s1의 Rouen·Sherbrooke 프레임은 UTx 데이터의 부분집합을 복사 없이 참조 (같은 원본·같은 압축 파일)
- 모든 run은 같은 세션의 같은 압축 파일·같은 보정 없음 기준으로 비교

## 검증
| 장면 | 지표 |
|---|---|
| St-Marc (주석 1000~1999, 4프레임 간격) | AP_gt·APs_gt (실제 주석, 클래스 무시), R. 2000/1000/500/250 kbps |
| René-Lévesque (주석 7200~8199, 4프레임 간격) | R만 (v8-2에서 모든 run의 AP_gt가 원본보다 높게 나옴) |

## 선택 규칙 (실행 전 고정)
UTx 또는 UTx_lb를 UT_s0 대신 고르는 조건 (모두 만족):
1. St-Marc AP_gt·APs_gt의 UT_s0 대비 차이가 500·250 kbps 모두 양수이고, 시드 차이(|UT_s1 − UT_s0|)보다 큼
2. René R_P3가 500·250 kbps에서 UT_s0보다 0.02 이상 낮지 않음
3. St-Marc 2000 kbps AP_gt가 보정 없음보다 1 이상 낮지 않음

둘 다 만족하면 APs_gt 500·250 평균이 높은 쪽. 아니면 U6_UT 설정 유지. 고른 설정 하나만 평가셋(v9)에서 확인.


In [ ]:
!pip install -q ultralytics==8.4.165

In [ ]:
%%writefile /kaggle/working/fcb_lib.py
"""FCB(주파수 기반 경량 Feature Correction Branch) 1280 공통 라이브러리
YOLO26s (Ultralytics 8.4.165) 고정, 입력 1280x720 -> letterbox 1280x736 (위아래 8px, 114)
"""
import json, subprocess, contextlib, io, time
import numpy as np, cv2, torch, torch.nn as nn, torch.nn.functional as F

W, H, PAD = 1280, 720, 8
HP = H + 2 * PAD                      # 736
GH, GW = HP // 8, W // 8              # DCT 격자 92 x 160 (= P3 격자)
CCTV = [1, 2, 3, 4, 6, 8]             # person, bicycle, car, motorcycle, bus, truck (COCO 80 인덱스 + 1)
BALL = [33]
SMALL = 32 ** 2

# ---------------- 디코딩 ----------------
def read_yuv(video, n=None):
    """yuv420p 디코딩 -> (Y[N,720,1280], U[N,360,640], V[N,360,640]) uint8"""
    p = subprocess.Popen(["ffmpeg", "-v", "error", "-i", video, "-f", "rawvideo", "-pix_fmt", "yuv420p", "-"],
                         stdout=subprocess.PIPE, stderr=subprocess.DEVNULL if n else None)
    fb = W * H * 3 // 2; Y, U, V = [], [], []
    while n is None or len(Y) < n:
        b = p.stdout.read(fb)
        if len(b) < fb: break
        a = np.frombuffer(b, np.uint8)
        Y.append(a[:W * H].reshape(H, W)); U.append(a[W * H:W * H * 5 // 4].reshape(H // 2, W // 2))
        V.append(a[W * H * 5 // 4:].reshape(H // 2, W // 2))
    p.stdout.close(); p.wait()
    return np.stack(Y), np.stack(U), np.stack(V)

def yuv_to_bgr(y, u, v):
    """OpenCV I420 -> BGR (사전실험 C·D와 같은 변환)"""
    return cv2.cvtColor(np.concatenate([y.ravel(), u.ravel(), v.ravel()]).reshape(H * 3 // 2, W), cv2.COLOR_YUV2BGR_I420)

# ---------------- GT (사전실험 C·D build_gt와 같은 규칙: 30fps 프레임 i = 원본 라벨 키 2i+1) ----------------
def build_gt(raw, n):
    """프레임별 [cat_id(1~80), x1, y1, x2, y2] (720p 좌표) 배열 리스트"""
    out = []
    for i in range(n):
        rows = []
        for o in raw.get(str(2 * i + 1), []):
            x1, y1, x2, y2 = o["x_min"] * W, o["y_min"] * H, o["x_max"] * W, o["y_max"] * H
            if x2 - x1 > 0 and y2 - y1 > 0: rows.append([int(o["class_id"]), x1, y1, x2, y2])
        out.append(np.array(rows, np.float32).reshape(-1, 5))
    return out

def coco_gt(gts, track=None):
    images, anns, aid = [], [], 0
    for i, g in enumerate(gts):
        images.append({"id": i, "width": W, "height": H})
        for c, x1, y1, x2, y2 in g:
            aid += 1; w, h = x2 - x1, y2 - y1
            anns.append({"id": aid, "image_id": i, "category_id": int(c), "bbox": [float(x1), float(y1), float(w), float(h)],
                         "area": float(w * h), "iscrowd": 0})
    return {"images": images, "annotations": anns, "categories": [{"id": k} for k in range(1, 81)]}

def coco_ap(gt, preds, classes, img_ids=None):
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    if not preds: return None
    with contextlib.redirect_stdout(io.StringIO()):
        cg = COCO(); cg.dataset = gt; cg.createIndex(); cd = cg.loadRes(preds)
        E = COCOeval(cg, cd, "bbox"); E.params.catIds = classes
        if img_ids is not None: E.params.imgIds = list(img_ids)
        E.evaluate(); E.accumulate(); E.summarize()
    f = lambda x: None if x < 0 else round(100 * x, 2)
    s = E.stats
    return {"AP": f(s[0]), "AP50": f(s[1]), "APs": f(s[3]), "APm": f(s[4]), "APl": f(s[5]), "ARs": f(s[9])}

# ---------------- 입력 텐서 ----------------
def frame_tensors(y, u, v, flip=False):
    """한 프레임 -> (rgb[3,736,1280] float 0~1, y[1,736,1280] float 0~255). Ultralytics LetterBox(114)·BGR->RGB와 동일"""
    if flip: y, u, v = y[:, ::-1], u[:, ::-1], v[:, ::-1]
    y, u, v = np.ascontiguousarray(y), np.ascontiguousarray(u), np.ascontiguousarray(v)
    bgr = cv2.copyMakeBorder(yuv_to_bgr(y, u, v), PAD, PAD, 0, 0, cv2.BORDER_CONSTANT, value=(114, 114, 114))
    rgb = torch.from_numpy(np.ascontiguousarray(bgr[..., ::-1].transpose(2, 0, 1))).float() / 255.0
    yp = torch.from_numpy(cv2.copyMakeBorder(y, PAD, PAD, 0, 0, cv2.BORDER_CONSTANT, value=114)).float()[None]
    return rgb, yp

def flip_boxes(g):
    g = g.copy(); x1 = W - g[:, 3]; g[:, 3] = W - g[:, 1]; g[:, 1] = x1; return g

def yolo_targets(gts):
    """프레임별 GT -> Ultralytics loss batch dict (1280x736 기준 정규화 xywh, cls 0~79)"""
    bi, cl, bb = [], [], []
    for j, g in enumerate(gts):
        if len(g) == 0: continue
        x1, y1, x2, y2 = g[:, 1], g[:, 2] + PAD, g[:, 3], g[:, 4] + PAD
        bi.append(np.full(len(g), j)); cl.append(g[:, 0] - 1)
        bb.append(np.stack([(x1 + x2) / 2 / W, (y1 + y2) / 2 / HP, (x2 - x1) / W, (y2 - y1) / HP], 1))
    if not bi: return {"batch_idx": torch.zeros(0), "cls": torch.zeros(0, 1), "bboxes": torch.zeros(0, 4)}
    return {"batch_idx": torch.from_numpy(np.concatenate(bi)).float(),
            "cls": torch.from_numpy(np.concatenate(cl)).float().view(-1, 1),
            "bboxes": torch.from_numpy(np.concatenate(bb)).float()}

def small_masks(gts, shapes):
    """작은 객체(원해상도 면적 < 32²) 박스와 겹치는 셀 = 1. shapes: [(h,w) per level], stride = HP/h"""
    out = []
    for (h, w) in shapes:
        s = HP / h; m = torch.zeros(len(gts), 1, h, w)
        for j, g in enumerate(gts):
            for c, x1, y1, x2, y2 in g:
                if (x2 - x1) * (y2 - y1) >= SMALL: continue
                c0, c1 = int(x1 // s), int(np.ceil(x2 / s)); r0, r1 = int((y1 + PAD) // s), int(np.ceil((y2 + PAD) / s))
                m[j, 0, r0:max(r1, r0 + 1), c0:max(c1, c0 + 1)] = 1
        out.append(m)
    return out

# ---------------- 8x8 블록 DCT (정규직교 DCT-II, scipy dctn norm='ortho'와 동일) ----------------
def dct_basis():
    x = np.arange(8); c = np.cos((2 * x[None] + 1) * x[:, None] * np.pi / 16) * np.sqrt(2 / 8); c[0] = np.sqrt(1 / 8)
    b = np.einsum("ux,vy->uvxy", c, c).reshape(64, 1, 8, 8)       # 채널 k = u*8+v (u: 세로 주파수, v: 가로)
    return torch.from_numpy(b).float()
UV = (np.arange(8)[:, None] + np.arange(8)[None]).ravel()
BANDS = {"DC": np.where(UV == 0)[0], "Low": np.where((UV >= 1) & (UV <= 2))[0],
         "Mid": np.where((UV >= 3) & (UV <= 6))[0], "High": np.where(UV >= 7)[0], "All": np.arange(64)}

class BlockDCT(nn.Module):
    def __init__(self):
        super().__init__(); self.register_buffer("basis", dct_basis())
    def forward(self, y):                    # (B,1,H,W) 0~255 -> (B,64,H/8,W/8)
        return F.conv2d(y, self.basis, stride=8)

def dct_np(y):                               # (H,W) -> (H/8, W/8, 64), 통계 계산용
    h, w = y.shape
    from scipy.fft import dctn
    c = dctn(y.astype(np.float64).reshape(h // 8, 8, w // 8, 8).transpose(0, 2, 1, 3), axes=(2, 3), norm="ortho")
    return c.reshape(h // 8, w // 8, 64)

# ---------------- FCB ----------------
class DWBlock(nn.Module):
    def __init__(self, c, stride=1, res=True):
        super().__init__(); self.res = res and stride == 1
        self.dw = nn.Conv2d(c, c, 3, stride, 1, groups=c); self.pw = nn.Conv2d(c, c, 1)
        self.gn = nn.GroupNorm(8, c); self.act = nn.SiLU()
    def forward(self, x):
        y = self.act(self.gn(self.pw(self.dw(x)))); return x + y if self.res else y

class FCB(nn.Module):
    """Y -> DCT -> 정규화 -> Stem -> Block x2 (F8) -> H3 (Δ3) / Down (F16) -> H4 (Δ4), Aux(F8 -> 목표 대역 ΔD)"""
    def __init__(self, mu, sigma, in_band="All", tgt_band="Mid", C=64, aux=True):
        super().__init__()
        self.in_idx = torch.as_tensor(BANDS[in_band]); self.tgt_idx = torch.as_tensor(BANDS[tgt_band])
        self.dct = BlockDCT()
        self.register_buffer("mu", torch.as_tensor(mu, dtype=torch.float32)[self.in_idx].view(1, -1, 1, 1))
        self.register_buffer("sigma", torch.as_tensor(sigma, dtype=torch.float32)[self.in_idx].view(1, -1, 1, 1))
        cin = len(self.in_idx)
        self.stem = nn.Sequential(nn.Conv2d(cin, C, 1), nn.GroupNorm(8, C), nn.SiLU())
        self.blocks = nn.Sequential(DWBlock(C), DWBlock(C))
        self.h3 = nn.Conv2d(C, 128, 1); self.down = DWBlock(C, stride=2); self.h4 = nn.Conv2d(C, 256, 1)
        for h in (self.h3, self.h4): nn.init.zeros_(h.weight); nn.init.zeros_(h.bias)
        self.aux = nn.Conv2d(C, len(self.tgt_idx), 1) if aux else None
    def forward(self, y):
        d = self.dct(y)[:, self.in_idx]
        f8 = self.blocks(self.stem((d - self.mu) / self.sigma))
        d3 = self.h3(f8); d4 = self.h4(self.down(f8))
        return d3, d4, (self.aux(f8) if (self.aux is not None and self.training) else None)
    def n_params(self, infer=True):
        return sum(p.numel() for n, p in self.named_parameters() if not (infer and n.startswith("aux")))

class CondFCB(nn.Module):
    """보정 모듈 변형 (탐지기 고정, Detect 직전 주입, 출력 0 초기화)
    arch="feat": 압축 P3·P4 feature만 입력 (DCT 없음)
    arch="cond": DCT(Y) 경로 + 압축 P3·P4 feature 합산 (설계 문서 FCB-cond)"""
    needs_feat = True
    def __init__(self, mu, sigma, arch="cond", in_band="All", tgt_band="Mid", C=64, aux=True):
        super().__init__(); self.arch = arch
        self.in_idx = torch.as_tensor(BANDS[in_band]); self.tgt_idx = torch.as_tensor(BANDS[tgt_band])
        gn = lambda: nn.GroupNorm(8, C)
        self.s3 = nn.Sequential(nn.Conv2d(128, C, 1), gn(), nn.SiLU()); self.s4 = nn.Sequential(nn.Conv2d(256, C, 1), gn(), nn.SiLU())
        if arch == "cond":
            self.dct = BlockDCT()
            self.register_buffer("mu", torch.as_tensor(mu, dtype=torch.float32)[self.in_idx].view(1, -1, 1, 1))
            self.register_buffer("sigma", torch.as_tensor(sigma, dtype=torch.float32)[self.in_idx].view(1, -1, 1, 1))
            self.stem = nn.Sequential(nn.Conv2d(len(self.in_idx), C, 1), gn(), nn.SiLU())
            self.down = DWBlock(C, stride=2)
        self.b8 = nn.Sequential(DWBlock(C), DWBlock(C)); self.b16 = nn.Sequential(DWBlock(C), DWBlock(C))
        self.h3 = nn.Conv2d(C, 128, 1); self.h4 = nn.Conv2d(C, 256, 1)
        for h in (self.h3, self.h4): nn.init.zeros_(h.weight); nn.init.zeros_(h.bias)
        self.aux = nn.Conv2d(C, len(self.tgt_idx), 1) if aux else None
    def forward(self, y, f3, f4):
        x8 = self.s3(f3.float())
        if self.arch == "cond": x8 = x8 + self.stem((self.dct(y)[:, self.in_idx] - self.mu) / self.sigma)
        f8 = self.b8(x8)
        x16 = self.s4(f4.float()) + (self.down(f8) if self.arch == "cond" else 0)
        f16 = self.b16(x16)
        return self.h3(f8), self.h4(f16), (self.aux(f8) if (self.aux is not None and self.training) else None)
    def n_params(self, infer=True):
        return sum(p.numel() for n, p in self.named_parameters() if not (infer and n.startswith("aux")))

def make_fcb(cfg, mu, sg):
    if cfg.get("arch", "fcb") == "fcb": return FCB(mu, sg, cfg["in_band"], cfg["tgt_band"], cfg["C"])
    return CondFCB(mu, sg, cfg["arch"], cfg["in_band"], cfg["tgt_band"], cfg["C"])

def run_fcb(fcb, y, f):
    return fcb(y, f[0], f[1]) if getattr(fcb, "needs_feat", False) else fcb(y)

# ---------------- 탐지기 ----------------
def load_detector(weights="yolo26s.pt", device="cuda"):
    from ultralytics import YOLO
    from ultralytics.cfg import get_cfg
    m = YOLO(weights).model.float().to(device)
    m.args = get_cfg()                                # loss gain(box 7.5, cls 0.5, dfl 1.5) 기본값
    for p in m.parameters(): p.requires_grad_(False)
    return m

def set_train_mode(m):
    """Detect 학습 출력(dict)을 쓰기 위해 train(), BatchNorm은 eval 고정"""
    m.train()
    for mod in m.modules():
        if isinstance(mod, nn.modules.batchnorm._BatchNorm): mod.eval()

def neck_feats(m, x):
    """Detect 직전까지 실행 -> [P3, P4, P5]"""
    y = []
    for mod in m.model[:-1]:
        if mod.f != -1: x = y[mod.f] if isinstance(mod.f, int) else [x if j == -1 else y[j] for j in mod.f]
        x = mod(x); y.append(x if mod.i in m.save else None)
    return [y[j] for j in m.model[-1].f]

def det_loss_fn(m):
    """one2many 손실만 (E2ELoss의 o2m/o2o 가중·감쇠 미사용)"""
    from ultralytics.utils.loss import v8DetectionLoss
    return v8DetectionLoss(m, tal_topk=10)

def l_det(crit, preds, batch):
    loss, items = crit(preds["one2many"], batch)
    return loss.sum() / preds["one2many"]["boxes"].shape[0], items

def rel_err(p, r, mask=None):
    """샘플별 ||p-r||² / ||r||², mask 있으면 마스크 영역만. 마스크 합 0인 샘플 제외. 반환 (평균, 유효 샘플 수)"""
    if mask is not None:
        mask = F.interpolate(mask, size=p.shape[-2:], mode="nearest") if mask.shape[-2:] != p.shape[-2:] else mask
        p, r = p * mask, r * mask
    num = ((p - r) ** 2).flatten(1).sum(1); den = (r ** 2).flatten(1).sum(1)
    ok = den > 0
    if ok.sum() == 0: return p.new_zeros(()), 0
    return (num[ok] / den[ok]).mean(), int(ok.sum())

# ---------------- 출력 증류 (원본 프레임 탐지기 출력 = 교사) ----------------
def kd_loss(s_out, t_out, anchor_w=None):
    """s_out/t_out: one2many dict (scores B×80×N 로짓, boxes B×4×N). 교사 최대 클래스 확률로 앵커 가중"""
    ts = torch.sigmoid(t_out["scores"].float()); ss = s_out["scores"].float()
    wt = ts.max(1).values                                                   # B×N 교사 확신도 (박스 가중)
    wc = torch.maximum(wt, torch.sigmoid(ss.detach()).max(1).values)          # 분류: 학생 오탐 앵커도 포함
    if anchor_w is not None: wt, wc = wt * anchor_w, wc * anchor_w
    bce = F.binary_cross_entropy_with_logits(ss, ts, reduction="none").sum(1)
    l1 = (s_out["boxes"].float() - t_out["boxes"].float()).abs().sum(1)
    return (wc * bce).sum() / (wc.sum() + 1.0), (wt * l1).sum() / (wt.sum() + 1.0)

def anchor_small_weight(gts, classes=None, boost=5.0):
    """작은 GT 박스(원해상도 면적 < 32²) 안 앵커에 boost배 가중. P3·P4·P5 순서로 펼침 (P5는 1)"""
    gg = [g if classes is None else g[np.isin(g[:, 0], classes)] for g in gts]
    ms = small_masks(gg, [(92, 160), (46, 80)])
    w = [1 + (boost - 1) * mm.flatten(1) for mm in ms] + [torch.ones(len(gts), 23 * 40)]
    return torch.cat(w, 1)

# ---------------- 추론 (Ultralytics predict와 같은 후처리) ----------------
@torch.no_grad()
def detect(m, rgb, fcb=None, y=None, conf=0.001, iou=0.7, max_det=300):
    """rgb (B,3,736,1280) -> 프레임별 [x,y,w,h,score,cat(1~80)] (720p 좌표). fcb 있으면 P3·P4에 Δ 더함"""
    from ultralytics.utils.nms import non_max_suppression
    m.eval()
    f = neck_feats(m, rgb)
    if fcb is not None:
        fcb.eval(); d3, d4, _ = run_fcb(fcb, y, f); f = [f[0] + d3, f[1] + d4, f[2]]
    return detect_from_feats(m, f, conf, iou, max_det)

@torch.no_grad()
def detect_from_feats(m, f, conf=0.001, iou=0.7, max_det=300):
    """[P3, P4, P5] -> Detect(eval) -> NMS -> 프레임별 [x,y,w,h,score,cat] (720p)"""
    from ultralytics.utils.nms import non_max_suppression
    m.eval(); out = m.model[-1](f)
    pr = non_max_suppression(out[0] if isinstance(out, tuple) else out, conf, iou, None, False, max_det=max_det, nc=0,
                             max_time_img=10.0)   # 기본 0.05(배치 8 -> 2.4초)면 conf 0.001에서 시간 초과로 검출이 잘림
    res = []
    for p in pr:
        p = p.cpu().numpy(); b = p[:, :4].copy(); b[:, [1, 3]] -= PAD
        b[:, [0, 2]] = b[:, [0, 2]].clip(0, W); b[:, [1, 3]] = b[:, [1, 3]].clip(0, H)
        res.append(np.column_stack([b[:, 0], b[:, 1], b[:, 2] - b[:, 0], b[:, 3] - b[:, 1], p[:, 4], p[:, 5] + 1]))
    return res

def to_coco(dets, ids):
    return [{"image_id": int(i), "category_id": int(r[5]), "bbox": [float(v) for v in r[:4]], "score": float(r[4])}
            for i, d in zip(ids, dets) for r in d]


In [ ]:
%%writefile /kaggle/working/fcb_train.py
# FCB 1280 ② 학습·평가: RUNS의 각 설정을 학습 -> 검증셋 평가 -> (정식 run이면) 평가셋 평가, 결과 CSV·가중치 저장
# 사용: RUNS = [dict(name=..., ...)] 정의 후 %run fcb_train.py  (기본값은 DEFAULT 참고)
import os, sys, glob, json, time, math, gc
import numpy as np, pandas as pd, torch
from concurrent.futures import ThreadPoolExecutor
POOL = ThreadPoolExecutor(4)   # cv2 색변환·패딩 병렬 (GIL 해제)
sys.path.insert(0, "/kaggle/working"); import fcb_lib as L
OUT = "/kaggle/working/fcb_runs"; os.makedirs(OUT, exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"; AMP = DEV == "cuda"
TRAIN_ALL = ["HighwayDrive", "WinterDrive", "TrafficLights", "OfficeWalk", "SpotRobot", "BoatReverse"]
EVAL = {"HighwayView": "cctv", "CampusView": "cctv", "VolleyballGame": "ball", "FloorballGame": "ball", "FloorballTrain": "ball"}
DEFAULT = dict(name="fcb", in_band="All", tgt_band="Mid", C=64, val_seq="WinterDrive",
               train_kbps=[2000, 1000, 500, 250], frame_step=1, epochs=20, batch=8, lr=1e-3, wd=1e-4, warmup=1,
               lam_det=1.0, lam_feat=1.0, lam_freq=1.0, flip=True, cache=False, seed=0,
               eval_val_kbps=[500, 250], eval_full=False, eval_kbps=["ref", 4000, 1000, 500, 250], ckpt=None, arch="fcb",
               lam_kd=0.0, kd_small=False, lam_reg=0.0)
SCREEN = dict(cache=True, flip=False, frame_step=2, train_kbps=[500, 250], epochs=10, eval_full=True, eval_kbps=[500, 250])
CACHE = {}   # 스크리닝 run끼리 같은 (학습 시퀀스, frame_step, 비트레이트)면 탐지기 feature 캐시 재사용
RUNS = globals().get("RUNS", [dict(name="smoke", epochs=1, frame_step=10)])
STAMP = time.strftime("%m%d_%H%M")

def find_dir(name):
    c = glob.glob(f"/kaggle/input/**/{name}", recursive=True); return c[0] if c else None
REF, GOP60, BDATA = find_dir("uvg_cctv"), find_dir("cctv_gop60"), find_dir("bdata")
def comp_path(seq, k):
    for root in (BDATA, GOP60):
        if root and os.path.exists(f"{root}/{seq}/cctv_{k}k.mp4"): return f"{root}/{seq}/cctv_{k}k.mp4"
    raise FileNotFoundError(f"{seq} {k}k")
def n_frames(s): return json.load(open(f"{REF}/{s}/done.json"))["ref_frames"]
def load_seq(s, kbs, ref=True):
    n = n_frames(s); d = {"comp": {k: L.read_yuv(comp_path(s, k), n) for k in kbs},
                         "gt": L.build_gt(json.load(open(f"{REF}/{s}/ann_raw.json")), n)}
    if ref: d["ref"] = L.read_yuv(f"{REF}/{s}/ref_720p30.mkv", n)
    return d

m = L.load_detector("yolo26s.pt", DEV); crit = L.det_loss_fn(m); dct = L.BlockDCT().to(DEV)
DATA = {}
def ensure(seqs, kbs):
    for s in seqs:
        if s not in DATA: DATA[s] = load_seq(s, kbs)
        for k in kbs:
            if k not in DATA[s]["comp"]: DATA[s]["comp"][k] = L.read_yuv(comp_path(s, k), n_frames(s))

def dct_stats(seqs, kbs):
    a = np.zeros((5, 64))
    with torch.no_grad():
        for s in seqs:
            R = DATA[s]["ref"][0]
            for k in kbs:
                C = DATA[s]["comp"][k][0]
                for i in range(0, len(R), 32):
                    yr = dct(torch.from_numpy(R[i:i + 32]).to(DEV).float()[:, None]).double()
                    yc = dct(torch.from_numpy(C[i:i + 32]).to(DEV).float()[:, None]).double(); dd = yr - yc
                    a[0] += yc.sum((0, 2, 3)).cpu().numpy(); a[1] += (yc ** 2).sum((0, 2, 3)).cpu().numpy()
                    a[2] += dd.sum((0, 2, 3)).cpu().numpy(); a[3] += (dd ** 2).sum((0, 2, 3)).cpu().numpy()
                    a[4] += yc.shape[0] * yc.shape[2] * yc.shape[3]
    n = a[4]; mu = a[0] / n; sg = np.sqrt(a[1] / n - mu ** 2); mud = a[2] / n; sgd = np.sqrt(a[3] / n - mud ** 2)
    return mu, sg, sgd

def make_batch(items, flip):
    fls = [flip and np.random.rand() < 0.5 for _ in items]
    jobs = [POOL.submit(L.frame_tensors, *(x[i] for x in DATA[s]["comp"][k]), flip=fl) for (s, i, k), fl in zip(items, fls)]
    jobs += [POOL.submit(L.frame_tensors, *(x[i] for x in DATA[s]["ref"]), flip=fl) for (s, i, k), fl in zip(items, fls)]
    res = [j.result() for j in jobs]; n = len(items)
    gts = [L.flip_boxes(DATA[s]["gt"][i]) if fl else DATA[s]["gt"][i] for (s, i, k), fl in zip(items, fls)]
    return (torch.stack([r[0] for r in res[:n]]), torch.stack([r[1] for r in res[:n]]),
            torch.stack([r[0] for r in res[n:]]), torch.stack([r[1] for r in res[n:]]), gts)

def build_cache(items):
    """탐지기 고정 + Detect 직전 주입 -> 압축 P3·P4·P5, 원본 P3·P4는 항목마다 고정. CPU에 fp16으로 저장"""
    C = {}; ref_done = {}
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
        m.eval()
        for j in range(0, len(items), 8):
            chunk = items[j:j + 8]; b = make_batch(chunk, flip=False)
            fc = L.neck_feats(m, b[0].to(DEV))
            fr = L.neck_feats(m, b[2].to(DEV))
            for q, it in enumerate(chunk):
                C[it] = [t[q].half().cpu() for t in fc]
                if (it[0], it[1]) not in ref_done: ref_done[(it[0], it[1])] = [t[q].half().cpu() for t in fr]
    return C, ref_done

def train_run(cfg):
    torch.manual_seed(cfg["seed"]); np.random.seed(cfg["seed"]); rng = np.random.default_rng(cfg["seed"])
    tr = [s for s in TRAIN_ALL if s != cfg["val_seq"]]
    ensure(tr + [cfg["val_seq"]], sorted(set(cfg["train_kbps"]) | set(cfg["eval_val_kbps"])))
    mu, sg, sgd = dct_stats(tr, cfg["train_kbps"])
    TB = torch.as_tensor(L.BANDS[cfg["tgt_band"]]); SGD = torch.as_tensor(sgd[TB.numpy()], dtype=torch.float32, device=DEV).view(1, -1, 1, 1)
    fcb = L.make_fcb(cfg, mu, sg).to(DEV); print("  params", fcb.n_params())
    frames = [(s, i) for s in tr for i in range(0, len(DATA[s]["gt"]), cfg["frame_step"])]
    steps_ep = max(1, len(frames) // cfg["batch"]); total = steps_ep * cfg["epochs"]; warm = steps_ep * cfg["warmup"]
    opt = torch.optim.AdamW(fcb.parameters(), cfg["lr"], weight_decay=cfg["wd"]); scaler = torch.amp.GradScaler(enabled=AMP)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda t: (t + 1) / warm if t < warm else 0.5 * (1 + math.cos(math.pi * (t - warm) / max(1, total - warm))))
    cache = refc = None
    if cfg["cache"]:
        key = (tuple(tr), cfg["frame_step"], tuple(cfg["train_kbps"]))
        if key not in CACHE:
            CACHE.clear(); gc.collect()
            t0 = time.time(); allit = [(s, i, k) for s, i in frames for k in cfg["train_kbps"]]
            CACHE[key] = build_cache(allit); print(f"  cache {len(CACHE[key][0])} items {time.time() - t0:.0f}s")
        cache, refc = CACHE[key]
    hist = []; t0 = time.time(); L.set_train_mode(m)
    for ep in range(cfg["epochs"]):
        order = rng.permutation(len(frames)); agg = {}
        for st in range(steps_ep):
            items = [(*frames[j], int(rng.choice(cfg["train_kbps"]))) for j in order[st * cfg["batch"]:(st + 1) * cfg["batch"]]]
            if cfg["cache"]:
                _, yc, _, yr, gts = make_batch(items, flip=False)                       # 캐시 모드: 반전 없음
                dt = torch.float16 if AMP else torch.float32
                fc = [torch.stack([cache[it][l] for it in items]).to(DEV, dt) for l in range(3)]
                fr = [torch.stack([refc[(it[0], it[1])][l] for it in items]).to(DEV, dt) for l in range(3)]
            else:
                rc, yc, rr, yr, gts = make_batch(items, cfg["flip"])
                with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
                    fc = L.neck_feats(m, rc.to(DEV)); fr = L.neck_feats(m, rr.to(DEV))
            yc, yr = yc.to(DEV), yr.to(DEV)
            with torch.no_grad(): tgt = ((dct(yr) - dct(yc))[:, TB] / SGD)[:, :, 1:-1]
            fcb.train(); L.set_train_mode(m)
            with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
                d3, d4, aux = L.run_fcb(fcb, yc, fc); p3, p4 = fc[0] + d3, fc[1] + d4
                preds = m.model[-1]([p3, p4, fc[2]])
            o = {k: (v.float() if torch.is_tensor(v) else [t.float() for t in v]) for k, v in preds["one2many"].items()}
            ld, _ = L.l_det(crit, {"one2many": o}, {k: v.to(DEV) for k, v in L.yolo_targets(gts).items()})
            p3, p4 = p3.float(), p4.float(); r3, r4 = fr[0].float(), fr[1].float()
            lg = L.rel_err(p3, r3)[0] + L.rel_err(p4, r4)[0]
            ms = [x.to(DEV) for x in L.small_masks(gts, [(92, 160), (46, 80)])]
            ls = L.rel_err(p3, r3, ms[0])[0] + L.rel_err(p4, r4, ms[1])[0]
            lf = ((aux.float()[:, :, 1:-1] - tgt) ** 2).mean() if aux is not None else torch.zeros((), device=DEV)
            lkc = lkb = lr = torch.zeros((), device=DEV)
            if cfg["lam_kd"] > 0:                                      # 교사: 같은 탐지기에 원본 feature
                with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
                    t_out = m.model[-1]([fr[0], fr[1], fr[2]])["one2many"]
                aw = L.anchor_small_weight(gts, L.CCTV + L.BALL).to(DEV) if cfg["kd_small"] else None
                lkc, lkb = L.kd_loss(o, t_out, aw)
            if cfg["lam_reg"] > 0:                                     # 보정량 크기 규제 (feature 에너지 대비)
                lr = (d3.float() ** 2).mean() / (fc[0].float() ** 2).mean() + (d4.float() ** 2).mean() / (fc[1].float() ** 2).mean()
            loss = (cfg["lam_det"] * ld + cfg["lam_feat"] * (lg + ls) + cfg["lam_freq"] * lf
                    + cfg["lam_kd"] * (lkc + lkb) + cfg["lam_reg"] * lr)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(fcb.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step()
            for k, v in {"L_det": ld, "L_glob": lg, "L_small": ls, "L_freq": lf, "L_kdc": lkc, "L_kdb": lkb, "L_reg": lr, "loss": loss}.items(): agg[k] = agg.get(k, 0) + v.item() / steps_ep
        agg["epoch"] = ep; agg["min"] = round((time.time() - t0) / 60, 2); hist.append(agg)
        print(f"  ep{ep} " + " ".join(f"{k} {v:.4f}" for k, v in agg.items() if k.startswith("L") or k == "loss") + f" | {agg['min']} min")
    return fcb, hist, (mu, sg, sgd)

def eval_seq(s, kbs, fcb=None, step=1, group=None, keep_preds=False):
    """평가 시퀀스 하나: 버전(k)별 COCO AP. 'ref' = 무압축 원본"""
    n = n_frames(s); gt = L.build_gt(json.load(open(f"{REF}/{s}/ann_raw.json")), n); ids = list(range(0, n, step))
    G = L.coco_gt(gt); rows, P = [], {}
    for k in kbs:
        Y, U, V = L.read_yuv(f"{REF}/{s}/ref_720p30.mkv", n) if k == "ref" else L.read_yuv(comp_path(s, k), n)
        dets = []
        for j in range(0, len(ids), 8):
            rgb, yp = map(torch.stack, zip(*POOL.map(lambda i: L.frame_tensors(Y[i], U[i], V[i]), ids[j:j + 8])))
            dets += L.detect(m, rgb.to(DEV), fcb, yp.to(DEV) if fcb is not None else None)
        pr = L.to_coco(dets, ids)
        for grp, cls in ([(group, L.CCTV if group == "cctv" else L.BALL)] if group else [("cctv", L.CCTV), ("ball", L.BALL)]):
            r = L.coco_ap(G, pr, cls, ids)
            if r: rows.append({"seq": s, "kbps": k, "group": grp, **r})
        if keep_preds: P[k] = np.concatenate([np.column_stack([np.full(len(d), i), d]) for i, d in zip(ids, dets)]).astype(np.float32)
        del Y, U, V
    return rows, P

def eval_groups(fcb, kbs, tag, keep_preds=True):
    """평가셋: 그룹(CCTV: HighwayView+CampusView / ball: 3개) 합쳐서 AP. 프레임 예측은 bootstrap용 저장"""
    rows = []
    for grp in ["cctv", "ball"]:
        seqs = [s for s, g in EVAL.items() if g == grp]; cls = L.CCTV if grp == "cctv" else L.BALL
        for k in kbs:
            imgs, anns, preds, off = [], [], [], 0
            for s in seqs:
                n = n_frames(s); gt = L.build_gt(json.load(open(f"{REF}/{s}/ann_raw.json")), n); G = L.coco_gt(gt)
                Y, U, V = L.read_yuv(f"{REF}/{s}/ref_720p30.mkv", n) if k == "ref" else L.read_yuv(comp_path(s, k), n)
                dets = []
                for j in range(0, n, 8):
                    rgb, yp = map(torch.stack, zip(*POOL.map(lambda i: L.frame_tensors(Y[i], U[i], V[i]), range(j, min(n, j + 8)))))
                    dets += L.detect(m, rgb.to(DEV), fcb, yp.to(DEV) if fcb is not None else None)
                imgs += [{**im, "id": im["id"] + off} for im in G["images"]]
                anns += [{**a, "id": a["id"] + off * 1000, "image_id": a["image_id"] + off} for a in G["annotations"]]
                preds += L.to_coco(dets, [i + off for i in range(n)])
                if keep_preds:
                    np.save(f"{OUT}/{tag}_preds_{s}_{k}.npy", np.concatenate([np.column_stack([np.full(len(d), i), d]) for i, d in enumerate(dets)]).astype(np.float32))
                off += 10000; del Y, U, V
            r = L.coco_ap({"images": imgs, "annotations": anns, "categories": G["categories"]}, preds, cls)
            rows.append({"run": tag, "group": grp, "kbps": k, **(r or {})}); print("  ", tag, grp, k, r)
    return rows

def oracle_eval(kbs=(500, 250), alphas=(0.25, 0.5, 1.0), tag="oracle"):
    """정답 보정량 주입 상한: P_l + α·(P_ref,l − P_comp,l). 학습 없음.
    설정: base(α=0), a{α} (P3·P4), a1_small (작은 GT 박스 셀만, α=1), a1_P3, a1_P4"""
    cfgs = {"base": (0, 1, 1, False)} | {f"a{a}": (a, 1, 1, False) for a in alphas} | \
           {"a1_small": (1, 1, 1, True), "a1_P3": (1, 1, 0, False), "a1_P4": (1, 0, 1, False)}
    rows = []
    for grp in ["cctv", "ball"]:
        seqs = [s for s, g in EVAL.items() if g == grp]; cls = L.CCTV if grp == "cctv" else L.BALL
        for k in kbs:
            imgs, anns, preds, off = [], [], {c: [] for c in cfgs}, 0
            for s in seqs:
                n = n_frames(s); gt = L.build_gt(json.load(open(f"{REF}/{s}/ann_raw.json")), n); G = L.coco_gt(gt)
                C = L.read_yuv(comp_path(s, k), n); R = L.read_yuv(f"{REF}/{s}/ref_720p30.mkv", n)
                dets = {c: [] for c in cfgs}
                for j in range(0, n, 8):
                    ids = list(range(j, min(n, j + 8)))
                    rc = torch.stack([r[0] for r in POOL.map(lambda i: L.frame_tensors(C[0][i], C[1][i], C[2][i]), ids)]).to(DEV)
                    rr = torch.stack([r[0] for r in POOL.map(lambda i: L.frame_tensors(R[0][i], R[1][i], R[2][i]), ids)]).to(DEV)
                    with torch.no_grad():
                        m.eval(); fc = L.neck_feats(m, rc); fr = L.neck_feats(m, rr)
                        ms = [x.to(DEV) for x in L.small_masks([gt[i] for i in ids], [(92, 160), (46, 80)])]
                        for c, (a, u3, u4, sm) in cfgs.items():
                            d3 = a * u3 * (fr[0] - fc[0]) * (ms[0] if sm else 1); d4 = a * u4 * (fr[1] - fc[1]) * (ms[1] if sm else 1)
                            dets[c] += L.detect_from_feats(m, [fc[0] + d3, fc[1] + d4, fc[2]])
                imgs += [{**im, "id": im["id"] + off} for im in G["images"]]
                anns += [{**a_, "id": a_["id"] + off * 1000, "image_id": a_["image_id"] + off} for a_ in G["annotations"]]
                for c in cfgs:
                    preds[c] += L.to_coco(dets[c], [i + off for i in range(n)])
                    np.save(f"{OUT}/{tag}_{c}_preds_{s}_{k}.npy", np.concatenate([np.column_stack([np.full(len(d), i), d]) for i, d in enumerate(dets[c])]).astype(np.float32))
                off += 10000; del C, R
            for c in cfgs:
                r = L.coco_ap({"images": imgs, "annotations": anns, "categories": G["categories"]}, preds[c], cls)
                rows.append({"run": f"{tag}_{c}", "group": grp, "kbps": k, **(r or {})}); print("  ", tag, c, grp, k, r)
    pd.DataFrame(rows).to_csv(f"{OUT}/oracle_{STAMP}.csv", index=False)
    return rows

# ---- 실행 ----
ALL = []
for run in RUNS:
    cfg = {**DEFAULT, **run}; tag = f"{cfg['name']}"
    print(f"=== {tag} ===", {k: v for k, v in cfg.items() if v != DEFAULT.get(k)})
    if cfg["name"] == "baseline":
        fcb, hist = None, []
    elif cfg["ckpt"]:                                   # 저장된 FCB 가중치로 평가만
        ck = torch.load(cfg["ckpt"], map_location=DEV); c0 = ck["cfg"]
        fcb = L.make_fcb({**DEFAULT, **c0}, np.zeros(64), np.ones(64)).to(DEV)
        fcb.load_state_dict(ck["state"]); hist = ck.get("hist", [])
        ensure([cfg["val_seq"]], cfg["eval_val_kbps"])
    else:
        fcb, hist, st = train_run(cfg)
        torch.save({"cfg": cfg, "state": fcb.state_dict(), "hist": hist}, f"{OUT}/{tag}.pt")
    vrows, _ = eval_seq(cfg["val_seq"], cfg["eval_val_kbps"], fcb, group="cctv")
    for r in vrows: r.update(run=tag, split="val")
    ALL += vrows; print("  val", [(r["kbps"], r["APs"]) for r in vrows])
    if cfg["eval_full"]:
        er = eval_groups(fcb, cfg["eval_kbps"], tag)
        for r in er: r["split"] = "eval"
        ALL += er
    pd.DataFrame(ALL).to_csv(f"{OUT}/results_{STAMP}.csv", index=False)
    json.dump({"cfg": cfg, "hist": hist}, open(f"{OUT}/{tag}_log.json", "w"), default=str)
print(pd.DataFrame(ALL).to_string())


In [ ]:
%%writefile /kaggle/working/fcb_v5.py
# FCB v5: 보정 위치 비교 (A 입력 영상 / B 백본 어댑터 / C 특징 공간 + 상위 문맥)
# 손실: 원본 feature 맞추기 (L_glob + L_small, P3·P4·P5). 라벨 손실 없음
# 검증: 2-fold (학습 시퀀스 6개를 3개씩 나눔). 지표 R = 1 - ||P'-P_ref||² / ||P-P_ref||² (학습에 안 쓴 시퀀스)
#       R=0: 보정 없음과 같음, R<0: 원본에서 더 멀어짐. 오라클 α=0.25 ≈ R 0.44
# 평가셋(HighwayView 등 5개)은 사용하지 않음
exec(open("/kaggle/working/fcb_train.py").read().split("# ---- " + "실행 ----")[0])
import torch.nn as nn, torch.nn.functional as F, contextlib, io, copy
KINDS = globals().get("KINDS", ["A", "B", "C"])
assert DEV == "cuda" or os.environ.get("V5_CPU_TEST"), "GPU 없음: Accelerator를 GPU T4로 설정"
OUT = "/kaggle/working/fcb_v5_runs"; os.makedirs(OUT, exist_ok=True)   # v4 결과(fcb_runs)와 분리

FOLDS = {"F1": ["TrafficLights", "OfficeWalk", "BoatReverse"], "F2": ["HighwayDrive", "WinterDrive", "SpotRobot"]}
V5 = dict(kbps=[2000, 1000, 500, 250], steps=1200, batch=4, lr=1e-3, wd=1e-4, warm=100, flip=True, seed=0,
          lam_small=1.0, eval_kbps=[2000, 500, 250], ap_kbps=[500, 250], eval_step=4)

def zero_(c): nn.init.zeros_(c.weight); nn.init.zeros_(c.bias); return c
class Dil(nn.Module):                       # depthwise 3×3 (dilation d) + 1×1 + GN + SiLU, residual
    def __init__(s, c, d):
        super().__init__(); s.dw = nn.Conv2d(c, c, 3, 1, d, dilation=d, groups=c); s.pw = nn.Conv2d(c, c, 1)
        s.gn = nn.GroupNorm(8, c); s.act = nn.SiLU()
    def forward(s, x): return x + s.act(s.gn(s.pw(s.dw(x))))
class Adapter(nn.Module):                   # x + out(blocks(in(x))), out 0 초기화
    def __init__(s, c, m, dils):
        super().__init__(); s.inp = nn.Sequential(nn.Conv2d(c, m, 1), nn.GroupNorm(8, m), nn.SiLU())
        s.blk = nn.Sequential(*[Dil(m, d) for d in dils]); s.out = zero_(nn.Conv2d(m, c, 1))
    def forward(s, x): return x + s.out(s.blk(s.inp(x)))

class ModA(nn.Module):                      # A-lite: RGB를 4×4 pixel-unshuffle(1/4 해상도, 48ch)에서 보정 -> RGB 잔차
    kind = "A"
    def __init__(s, m=48, dils=(1, 2, 4)): super().__init__(); s.ad = Adapter(48, m, dils)
    def forward(s, rgb): return F.pixel_shuffle(s.ad(F.pixel_unshuffle(rgb, 4)), 4)
class ModB(nn.Module):                      # B: 백본 layer 2(stride 4, 128ch)·layer 4(stride 8, 256ch) 출력에 어댑터
    kind = "B"
    def __init__(s, m2=24, m4=48, dils=(1, 2)):
        super().__init__(); s.at = nn.ModuleDict({"2": Adapter(128, m2, dils), "4": Adapter(256, m4, dils)})
class ModC(nn.Module):                      # C: P3 격자에서 P3 + P4·P5(위에서 내려준 문맥) -> ΔP3, ΔP4
    kind = "C"
    def __init__(s, m=64, dils=(1, 2, 4)):
        super().__init__(); s.r3, s.r4, s.r5 = nn.Conv2d(128, m, 1), nn.Conv2d(256, m, 1), nn.Conv2d(512, m, 1)
        s.norm = nn.Sequential(nn.GroupNorm(8, m), nn.SiLU()); s.blk = nn.Sequential(*[Dil(m, d) for d in dils])
        s.h3 = zero_(nn.Conv2d(m, 128, 1)); s.down = L.DWBlock(m, stride=2, res=False); s.h4 = zero_(nn.Conv2d(m, 256, 1))
    def forward(s, f):
        p3, p4, p5 = f; sz = p3.shape[-2:]
        z = s.r3(p3) + F.interpolate(s.r4(p4), size=sz, mode="nearest") + F.interpolate(s.r5(p5), size=sz, mode="nearest")
        z = s.blk(s.norm(z)); return [p3 + s.h3(z), p4 + s.h4(s.down(z)), p5]
MODS = {"A": ModA, "B": ModB, "C": ModC}

def feats(rgb, mod=None):
    """보정 적용한 Detect 입력 [P3, P4, P5]. mod None = 보정 없음"""
    if mod is None or mod.kind == "C":
        with torch.no_grad(): f = L.neck_feats(m, rgb)
        return f if mod is None else mod(f)
    x = mod(rgb) if mod.kind == "A" else rgb; y = []
    for l in m.model[:-1]:
        if l.f != -1: x = y[l.f] if isinstance(l.f, int) else [x if j == -1 else y[j] for j in l.f]
        x = l(x)
        if mod.kind == "B" and str(l.i) in mod.at: x = mod.at[str(l.i)](x)
        y.append(x if l.i in m.save else None)
    return [y[j] for j in m.model[-1].f]

def cost(mod):
    """params, GFLOPs(=2×MAC, 1280×736 입력 1장, thop)"""
    import thop
    n = sum(p.numel() for p in mod.parameters()); mm = mod.to("cpu").eval()
    if mod.kind == "A": ins = [(mm, (torch.zeros(1, 3, 736, 1280),))]
    elif mod.kind == "B": ins = [(mm.at["2"], (torch.zeros(1, 128, 184, 320),)), (mm.at["4"], (torch.zeros(1, 256, 92, 160),))]
    else: ins = [(mm, ([torch.zeros(1, 128, 92, 160), torch.zeros(1, 256, 46, 80), torch.zeros(1, 512, 23, 40)],))]
    g = 0.0
    try:
        for mo, x in ins:
            with contextlib.redirect_stdout(io.StringIO()): g += thop.profile(copy.deepcopy(mo), inputs=x, verbose=False)[0]
        g = round(2 * g / 1e9, 3)
    except Exception as e: print("  thop 실패:", e); g = None
    finally: mod.to(DEV)
    return n, g

def train5(kind, val, cfg=V5):
    torch.manual_seed(cfg["seed"]); np.random.seed(cfg["seed"]); rng = np.random.default_rng(cfg["seed"])   # np.random: 반전 여부
    tr = [s for s in TRAIN_ALL if s not in val]; t0 = time.time(); ensure(tr, cfg["kbps"])
    print(f"  {kind} 학습 시작: 학습 시퀀스 {tr} (데이터 로딩 {time.time() - t0:.0f}s)", flush=True)
    mod = MODS[kind]().to(DEV); frames = [(s, i) for s in tr for i in range(len(DATA[s]["gt"]))]
    opt = torch.optim.AdamW(mod.parameters(), cfg["lr"], weight_decay=cfg["wd"]); scaler = torch.amp.GradScaler(enabled=AMP)
    T, W = cfg["steps"], cfg["warm"]
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda t: (t + 1) / W if t < W else 0.5 * (1 + math.cos(math.pi * (t - W) / max(1, T - W))))
    hist, agg, t0 = [], {}, time.time(); m.eval()
    for st in range(T):
        items = [(*frames[j], int(rng.choice(cfg["kbps"]))) for j in rng.choice(len(frames), cfg["batch"], replace=False)]
        rc, _, rr, _, gts = make_batch(items, cfg["flip"]); rc, rr = rc.to(DEV), rr.to(DEV)
        mod.train()
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
            with torch.no_grad(): fr = L.neck_feats(m, rr)
            f = feats(rc, mod)
        f, fr = [x.float() for x in f], [x.float() for x in fr]
        ms = [x.to(DEV) for x in L.small_masks(gts, [(92, 160), (46, 80)])]
        lg = sum(L.rel_err(a, b)[0] for a, b in zip(f, fr)); ls = L.rel_err(f[0], fr[0], ms[0])[0] + L.rel_err(f[1], fr[1], ms[1])[0]
        loss = lg + cfg["lam_small"] * ls
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(mod.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step()
        for k, v in {"L_glob": lg, "L_small": ls}.items(): agg[k] = agg.get(k, 0) + v.item()
        if (st + 1) % 200 == 0:
            r = {k: round(v / 200, 4) for k, v in agg.items()}; r.update(step=st + 1, min=round((time.time() - t0) / 60, 2))
            hist.append(r); agg = {}; print("   ", r, flush=True)
    return mod, hist

@torch.no_grad()
def eval5(mod, val, cfg=V5):
    """학습에 안 쓴 시퀀스: 비트레이트별 R(P3·P4·P5·P3 작은 객체 영역) + CCTV APs(묶어서). mod None = 보정 없음(AP만 의미)"""
    ensure(val, cfg["eval_kbps"]); rows = []
    if mod is not None: mod.eval()
    for k in cfg["eval_kbps"]:
        num, den = np.zeros(4), np.zeros(4); imgs, anns, preds, off, G = [], [], [], 0, None
        for s in val:
            n = len(DATA[s]["gt"]); ids = list(range(0, n, cfg["eval_step"])); G = L.coco_gt(DATA[s]["gt"]); dets = []
            for j in range(0, len(ids), 8):
                items = [(s, i, k) for i in ids[j:j + 8]]; rc, _, rr, _, gts = make_batch(items, flip=False)
                fc = L.neck_feats(m, rc.to(DEV)); fr = L.neck_feats(m, rr.to(DEV)); f = fc if mod is None else feats(rc.to(DEV), mod)
                ms = L.small_masks(gts, [(92, 160)])[0].to(DEV)
                for q in range(3):
                    num[q] += ((f[q] - fr[q]) ** 2).sum().item(); den[q] += ((fc[q] - fr[q]) ** 2).sum().item()
                num[3] += (((f[0] - fr[0]) * ms) ** 2).sum().item(); den[3] += (((fc[0] - fr[0]) * ms) ** 2).sum().item()
                if k in cfg["ap_kbps"]: dets += L.detect_from_feats(m, f)
            if k in cfg["ap_kbps"]:
                imgs += [{**im, "id": im["id"] + off} for im in G["images"] if im["id"] in ids]
                anns += [{**a, "id": a["id"] + off * 1000, "image_id": a["image_id"] + off} for a in G["annotations"] if a["image_id"] in ids]
                preds += L.to_coco(dets, [i + off for i in ids])
            off += 10000
        R = 1 - num / np.maximum(den, 1e-12)
        row = {"kbps": k, "R_P3": R[0], "R_P4": R[1], "R_P5": R[2], "R_P3small": R[3]}
        if k in cfg["ap_kbps"]:
            ap = L.coco_ap({"images": imgs, "annotations": anns, "categories": G["categories"]}, preds, L.CCTV)
            row.update({kk: ap[kk] for kk in ("AP", "APs", "APm", "APl")} if ap else {})
        rows.append({kk: (round(v, 4) if isinstance(v, float) else v) for kk, v in row.items()})
    return rows

# ==== v5 실행 ====
RES5 = []; t_all = time.time()
for fold, val in FOLDS.items():
    print(f"##### {fold} 검증 {val} #####", flush=True); t0 = time.time()
    for r in eval5(None, val): RES5.append({"fold": fold, "run": "none", **r}); print("  none", r, flush=True)
    print(f"  검증 데이터 로딩·보정 없음 평가 {(time.time() - t0) / 60:.1f}분", flush=True)
    for kind in KINDS:
        t0 = time.time(); mod, hist = train5(kind, val); n, g = cost(mod)
        torch.save({"kind": kind, "state": mod.state_dict(), "hist": hist, "V5": V5}, f"{OUT}/v5_{kind}_{fold}.pt")
        for r in eval5(mod, val): RES5.append({"fold": fold, "run": kind, "params": n, "GFLOPs": g, **r}); print(" ", kind, r, flush=True)
        print(f"  {kind} {fold} 끝 {(time.time() - t0) / 60:.1f}분 (누적 {(time.time() - t_all) / 60:.1f}분)", flush=True)
        pd.DataFrame(RES5).to_csv(f"{OUT}/v5_results_{STAMP}.csv", index=False)
        del mod; torch.cuda.empty_cache()
print(pd.DataFrame(RES5).to_string())


In [ ]:
%%writefile /kaggle/working/fcb_v6.py
# FCB v6: B(백본 어댑터) 개선 비교 + 고정 CCTV 검증 (Urban Tracker St-Marc)
# 학습: UVG 학습 시퀀스 6개 전부. 검증: St-Marc (학습에 안 씀). 평가셋 5개는 사용하지 않음
# St-Marc: 원 사이트에서 노트북 안에서 다운로드 (재업로드 없음). 원본 = 배포 영상(H.264 6.2Mbps) 그대로 디코딩
# 지표: R(P3·P4·P5·P3 작은 객체 영역), AP_pseudo(원본 프레임 탐지 결과 conf 0.25를 정답으로), AP_gt(St-Marc 주석, 클래스 무시)
exec(open("/kaggle/working/fcb_v5.py").read().split("# ==== v5 실행 ====")[0])
import sqlite3, zipfile, urllib.request, subprocess

UT = "/kaggle/working/urbantracker"; os.makedirs(UT, exist_ok=True)
OUT = "/kaggle/working/fcb_v6_runs"; os.makedirs(OUT, exist_ok=True)
V6 = dict(V5, steps=1500, eval_kbps=[2000, 500, 250], ap_kbps=[2000, 500, 250], eval_step=4, n_val=1000, lam_det=0.0)
URL = "https://urbantracker.jpjodoin.com/dataset/stmarc"

# ---------------- St-Marc 준비 ----------------
def get(name):
    p = f"{UT}/{name}"
    if not os.path.exists(p): urllib.request.urlretrieve(f"{URL}/{name}", p)
    return p

def parse_polytrack(zp):
    """Polytrack 주석 zip -> {frame: [[1, x1, y1, x2, y2], ...]} (클래스 무시). 스키마 출력. 실패 시 None"""
    try:
        d = f"{UT}/ann"; zipfile.ZipFile(zp).extractall(d)
        dbs = [os.path.join(r, f) for r, _, fs in os.walk(d) for f in fs if f.lower().endswith((".sqlite", ".db", ".sqlite3"))]
        print("  주석 파일:", [os.path.relpath(x, d) for r, _, fs in os.walk(d) for x in [os.path.join(r, f) for f in fs]])
        if not dbs: return None
        con = sqlite3.connect(dbs[0]); cur = con.cursor()
        for (t,) in cur.execute("select name from sqlite_master where type='table'").fetchall():
            cols = [c[1] for c in cur.execute(f"pragma table_info({t})").fetchall()]
            print(f"  table {t}: {cols} rows={cur.execute(f'select count(*) from {t}').fetchone()[0]}")
        rows = cur.execute("select frame_number, x_top_left, y_top_left, x_bottom_right, y_bottom_right from bounding_boxes").fetchall()
        try: print("  road_user_type 분포:", cur.execute("select road_user_type, count(*) from objects group by road_user_type").fetchall())
        except Exception as e: print("  objects 테이블 없음:", e)
        g = {}
        for f, x1, y1, x2, y2 in rows: g.setdefault(int(f), []).append([1, min(x1, x2), min(y1, y2), max(x1, x2), max(y1, y2)])
        fs = sorted(g); print(f"  주석 프레임 {fs[0]}~{fs[-1]} ({len(fs)}개), 박스 {len(rows)}개")
        return g
    except Exception as e:
        print("  주석 파싱 실패:", repr(e)); return None

def encode_cctv(ref, kbps, out, gop=60):            # A-3·B 데이터와 같은 명령
    if os.path.exists(out): return
    cmd = (f'ffmpeg -y -loglevel error -i "{ref}" -c:v libx264 -preset veryfast -tune zerolatency '
           f'-profile:v high -pix_fmt yuv420p -b:v {kbps}k -maxrate {kbps}k -bufsize {kbps}k '
           f'-g {gop} -keyint_min {gop} -sc_threshold 0 -an "{out}"')
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode: raise RuntimeError(r.stderr[-2000:])

def prep_stmarc(cfg=V6):
    import subprocess as sp
    t0 = time.time(); vid = get("stmarc_video.avi")
    try: gt_raw = parse_polytrack(get("stmarc_annotations.zip"))
    except Exception as e: print("  주석 다운로드 실패:", repr(e)); gt_raw = None
    start = min(gt_raw) if gt_raw else 0
    n = cfg["n_val"] if not gt_raw else min(cfg["n_val"], max(gt_raw) - start + 1)
    ref = f"{UT}/stmarc_ref.mkv"
    if not os.path.exists(ref):
        r = sp.run(f'ffmpeg -y -loglevel error -i "{vid}" -vf "select=between(n\\,{start}\\,{start + n - 1})" -vsync 0 '
                   f'-c:v ffv1 -pix_fmt yuv420p -an "{ref}"', shell=True, capture_output=True, text=True)
        if r.returncode: raise RuntimeError(r.stderr[-2000:])
    for k in cfg["eval_kbps"]: encode_cctv(ref, k, f"{UT}/stmarc_{k}k.mp4")
    d = {"ref": L.read_yuv(ref, n), "comp": {k: L.read_yuv(f"{UT}/stmarc_{k}k.mp4", n) for k in cfg["eval_kbps"]}}
    n = len(d["ref"][0]); ids = list(range(0, n, cfg["eval_step"]))
    # 의사 정답: 원본 프레임에 고정 탐지기 (conf 0.25, CCTV 클래스) -> [cat, x1, y1, x2, y2]
    pg = [np.zeros((0, 5)) for _ in range(n)]
    for j in range(0, len(ids), 8):
        b = [L.frame_tensors(*(x[i] for x in d["ref"]))[0] for i in ids[j:j + 8]]
        for i, dt in zip(ids[j:j + 8], L.detect(m, torch.stack(b).to(DEV), conf=0.25)):
            dt = dt[np.isin(dt[:, 5], L.CCTV)]
            pg[i] = np.column_stack([dt[:, 5], dt[:, 0], dt[:, 1], dt[:, 0] + dt[:, 2], dt[:, 1] + dt[:, 3]]) if len(dt) else np.zeros((0, 5))
    d["gt"] = pg
    d["gt_real"] = [np.array(gt_raw.get(start + i, []), float).reshape(-1, 5) for i in range(n)] if gt_raw else None
    a = np.concatenate([pg[i] for i in ids]); sm = ((a[:, 3] - a[:, 1]) * (a[:, 4] - a[:, 2]) < L.SMALL).sum()
    print(f"  St-Marc 프레임 {start}~{start + n - 1} ({n}장), 평가 {len(ids)}장, 의사 정답 {len(a)}개 (작은 객체 {sm}), {time.time() - t0:.0f}s")
    if d["gt_real"]:
        r = np.concatenate([d["gt_real"][i] for i in ids]); print(f"  실제 주석 {len(r)}개 (작은 객체 {((r[:, 3] - r[:, 1]) * (r[:, 4] - r[:, 2]) < L.SMALL).sum()})")
    DATA["StMarc"] = d

# ---------------- B 변형: 비트레이트 조건(FiLM) ----------------
class AdapterF(Adapter):                   # cond=False면 v5 Adapter와 같음 (FiLM 없음)
    def __init__(s, c, m, dils, de=16, cond=False):
        super().__init__(c, m, dils)
        if cond: s.film = zero_(nn.Linear(de, 2 * m))   # 0 초기화: 시작 시 비트레이트 무시
    def forward(s, x, e=None):
        h = s.inp(x)
        if e is not None: g, b = s.film(e).chunk(2, 1); h = h * (1 + g[:, :, None, None]) + b[:, :, None, None]
        return x + s.out(s.blk(h))
class ModB6(nn.Module):
    kind = "B"
    def __init__(s, cond=False, m2=24, m4=48, dils=(1, 2), de=16):
        super().__init__(); s.cond = cond
        s.at = nn.ModuleDict({"2": AdapterF(128, m2, dils, de, cond), "4": AdapterF(256, m4, dils, de, cond)})
        if cond: s.emb = nn.Sequential(nn.Linear(1, de), nn.SiLU(), nn.Linear(de, de), nn.SiLU())
    def e(s, kb): return s.emb(torch.log2(kb.float() / 500.0)[:, None]) if s.cond else None

def feats6(rgb, mod=None, kb=None):
    if mod is None or mod.kind != "B": return feats(rgb, mod)
    e = mod.e(kb.to(rgb.device)) if mod.cond else None; x, y = rgb, []
    for l in m.model[:-1]:
        if l.f != -1: x = y[l.f] if isinstance(l.f, int) else [x if j == -1 else y[j] for j in l.f]
        x = l(x)
        if str(l.i) in mod.at: x = mod.at[str(l.i)](x, e)
        y.append(x if l.i in m.save else None)
    return [y[j] for j in m.model[-1].f]

def make_mod(spec):
    if spec["kind"] == "B": return ModB6(cond=spec.get("cond", False))
    return MODS[spec["kind"]]()

# ---------------- 학습 (UVG 6개 전부) ----------------
def train6(spec, cfg=V6):
    cfg = {**cfg, **spec}; torch.manual_seed(cfg["seed"]); np.random.seed(cfg["seed"]); rng = np.random.default_rng(cfg["seed"])
    t0 = time.time(); ensure(TRAIN_ALL, cfg["kbps"]); print(f"  {spec['name']} 학습 시작 (데이터 로딩 {time.time() - t0:.0f}s)", flush=True)
    mod = make_mod(spec).to(DEV); frames = [(s, i) for s in TRAIN_ALL for i in range(len(DATA[s]["gt"]))]
    opt = torch.optim.AdamW(mod.parameters(), cfg["lr"], weight_decay=cfg["wd"]); scaler = torch.amp.GradScaler(enabled=AMP)
    T, W = cfg["steps"], cfg["warm"]
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda t: (t + 1) / W if t < W else 0.5 * (1 + math.cos(math.pi * (t - W) / max(1, T - W))))
    hist, agg, t0 = [], {}, time.time()
    for st in range(T):
        items = [(*frames[j], int(rng.choice(cfg["kbps"]))) for j in rng.choice(len(frames), cfg["batch"], replace=False)]
        rc, _, rr, _, gts = make_batch(items, cfg["flip"]); rc, rr = rc.to(DEV), rr.to(DEV)
        kb = torch.tensor([it[2] for it in items], device=DEV)
        mod.train(); m.eval()
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
            with torch.no_grad(): fr = L.neck_feats(m, rr)
            f = feats6(rc, mod, kb)
            ld = torch.zeros((), device=DEV)
            if cfg["lam_det"] > 0:                      # 탐지 손실 (one2many, Detect train 모드, BN eval)
                L.set_train_mode(m); o = m.model[-1](f)["one2many"]; m.eval()
                o = {k: (v.float() if torch.is_tensor(v) else [t.float() for t in v]) for k, v in o.items()}
        if cfg["lam_det"] > 0:
            ld, _ = L.l_det(crit, {"one2many": o}, {k: v.to(DEV) for k, v in L.yolo_targets(gts).items()})
        f, fr = [x.float() for x in f], [x.float() for x in fr]
        ms = [x.to(DEV) for x in L.small_masks(gts, [(92, 160), (46, 80)])]
        lg = sum(L.rel_err(a, b)[0] for a, b in zip(f, fr)); ls = L.rel_err(f[0], fr[0], ms[0])[0] + L.rel_err(f[1], fr[1], ms[1])[0]
        loss = lg + cfg["lam_small"] * ls + cfg["lam_det"] * ld
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(mod.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step()
        for k, v in {"L_glob": lg, "L_small": ls, "L_det": ld}.items(): agg[k] = agg.get(k, 0) + v.item()
        if (st + 1) % 250 == 0:
            r = {k: round(v / 250, 4) for k, v in agg.items()}; r.update(step=st + 1, min=round((time.time() - t0) / 60, 2))
            hist.append(r); agg = {}; print("   ", r, flush=True)
    return mod, hist

# ---------------- St-Marc 검증 ----------------
def ap_rows(gts, dets, ids, classes, agnostic=False):
    if agnostic:
        gts = [g.copy() for g in gts]; dets = [d[np.isin(d[:, 5], L.CCTV)].copy() for d in dets]
        for g in gts: g[:, 0] = 1
        for d in dets: d[:, 5] = 1
    G = L.coco_gt(gts); G["images"] = [im for im in G["images"] if im["id"] in set(ids)]
    G["annotations"] = [a for a in G["annotations"] if a["image_id"] in set(ids)]
    full = [np.zeros((0, 6))] * len(gts)
    for i, d in zip(ids, dets): full[i] = d
    return L.coco_ap(G, L.to_coco([full[i] for i in ids], ids), [1] if agnostic else classes)

@torch.no_grad()
def eval6(mod, cfg=V6):
    d = DATA["StMarc"]; n = len(d["ref"][0]); ids = list(range(0, n, cfg["eval_step"])); rows = []
    if mod is not None: mod.eval()
    m.eval()
    for k in cfg["eval_kbps"]:
        num, den = np.zeros(4), np.zeros(4); dets = []
        for j in range(0, len(ids), 8):
            items = [("StMarc", i, k) for i in ids[j:j + 8]]; rc, _, rr, _, gts = make_batch(items, flip=False)
            rc, rr = rc.to(DEV), rr.to(DEV); kb = torch.full((len(items),), k, device=DEV)
            fc = L.neck_feats(m, rc); fr = L.neck_feats(m, rr); f = fc if mod is None else feats6(rc, mod, kb)
            ms = L.small_masks(gts, [(92, 160)])[0].to(DEV)
            for q in range(3):
                num[q] += ((f[q] - fr[q]) ** 2).sum().item(); den[q] += ((fc[q] - fr[q]) ** 2).sum().item()
            num[3] += (((f[0] - fr[0]) * ms) ** 2).sum().item(); den[3] += (((fc[0] - fr[0]) * ms) ** 2).sum().item()
            dets += L.detect_from_feats(m, f)
        R = 1 - num / np.maximum(den, 1e-12)
        row = {"kbps": k, "R_P3": R[0], "R_P4": R[1], "R_P5": R[2], "R_P3small": R[3]}
        ap = ap_rows(d["gt"], dets, ids, L.CCTV)
        if ap: row.update({f"{kk}_pseudo": ap[kk] for kk in ("AP", "APs", "APm", "APl")})
        if d["gt_real"] is not None:
            ap = ap_rows(d["gt_real"], dets, ids, None, agnostic=True)
            if ap: row.update({f"{kk}_gt": ap[kk] for kk in ("AP", "AP50", "APs", "APm", "APl")})
        rows.append({kk: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for kk, v in row.items()})
    return rows

# ==== v6 실행 ====
RUNS6 = globals().get("RUNS6", [
    dict(name="A", kind="A"),
    dict(name="B", kind="B"),
    dict(name="B_cond", kind="B", cond=True),
    dict(name="B_det", kind="B", lam_det=0.1),
    dict(name="B_small", kind="B", lam_small=4.0),
])
RES6 = []; t_all = time.time()
print("##### St-Marc 준비 #####", flush=True); prep_stmarc()
for r in eval6(None): RES6.append({"run": "none", **r}); print("  none", r, flush=True)
pd.DataFrame(RES6).to_csv(f"{OUT}/v6_results_{STAMP}.csv", index=False)
for spec in RUNS6:
    t0 = time.time(); mod, hist = train6(spec); n_p, g = cost(mod)
    torch.save({"spec": spec, "state": mod.state_dict(), "hist": hist, "V6": V6}, f"{OUT}/v6_{spec['name']}.pt")
    for r in eval6(mod): RES6.append({"run": spec["name"], "params": n_p, "GFLOPs": g, **r}); print(" ", spec["name"], r, flush=True)
    print(f"  {spec['name']} 끝 {(time.time() - t0) / 60:.1f}분 (누적 {(time.time() - t_all) / 60:.1f}분)", flush=True)
    pd.DataFrame(RES6).to_csv(f"{OUT}/v6_results_{STAMP}.csv", index=False)
    del mod; torch.cuda.empty_cache()
print(pd.DataFrame(RES6).to_string())


In [ ]:
%%writefile /kaggle/working/fcb_v7.py
# FCB v7: B_small 기반, 손실 정규화(보정 없음 오차 기준) + 비트레이트 조건 비교. 검증 St-Marc (전체 주석)
# v6 대비: ① 주석 파일 stmarc_gt.sqlite(전체) 사용 ② 주석 프레임 번호 오프셋 자동 확인 ③ 손실 "R" 추가
#          ④ 평가 4000/2000/1000/500/250 kbps ⑤ 예측·정답 저장(부트스트랩용) ⑥ 평가 프레임만 메모리에 유지
# 평가셋 5개는 사용하지 않음
exec(open("/kaggle/working/fcb_v6.py").read().split("# ==== v6 실행 ====")[0])

OUT = "/kaggle/working/fcb_v7_runs"; os.makedirs(OUT, exist_ok=True)
V7 = dict(V6, steps=1500, lam_small=4.0, loss="rel", eval_kbps=[4000, 2000, 1000, 500, 250], eval_step=4, n_val=1000)

def parse_polytrack(zp, seq="stmarc"):
    """전체 주석 sqlite(<seq>_gt.sqlite) -> {frame: [[1, x1, y1, x2, y2], ...]} (클래스 무시)"""
    try:
        d = f"{UT}/ann"; zipfile.ZipFile(zp).extractall(d)
        files = [os.path.join(r, f) for r, _, fs in os.walk(d) for f in fs]
        print("  주석 파일:", [os.path.relpath(x, d) for x in files])
        for x in files:
            if x.lower().endswith("readme.txt"): print("  --- ReadMe ---\n" + open(x, errors="ignore").read()[:1500] + "\n  -------------")
        db = [x for x in files if os.path.basename(x).lower() == f"{seq}_gt.sqlite"]
        if not db: print("  전체 주석 sqlite 없음"); return None
        con = sqlite3.connect(db[0]); cur = con.cursor(); print("  사용:", os.path.basename(db[0]))
        for (t,) in cur.execute("select name from sqlite_master where type='table'").fetchall():
            cols = [c[1] for c in cur.execute(f"pragma table_info({t})").fetchall()]
            print(f"  table {t}: {cols} rows={cur.execute(f'select count(*) from {t}').fetchone()[0]}")
        for q, lab in [("select * from objects_type", "objects_type"),
                       ("select road_user_type, count(*) from objects group by road_user_type", "road_user_type 분포")]:
            try: print(f"  {lab}:", cur.execute(q).fetchall())
            except Exception as e: print(f"  {lab} 없음:", e)
        rows = cur.execute("select frame_number, x_top_left, y_top_left, x_bottom_right, y_bottom_right from bounding_boxes").fetchall()
        g = {}
        for f, x1, y1, x2, y2 in rows: g.setdefault(int(f), []).append([1, min(x1, x2), min(y1, y2), max(x1, x2), max(y1, y2)])
        fs = sorted(g); print(f"  주석 프레임 {fs[0]}~{fs[-1]} ({len(fs)}개), 박스 {len(rows)}개")
        return g
    except Exception as e:
        print("  주석 파싱 실패:", repr(e)); return None

def iou_match(a, b):
    """a 박스마다 b와의 최대 IoU 평균 (a, b: [., x1, y1, x2, y2])"""
    if len(a) == 0 or len(b) == 0: return None
    A, B = a[:, None, 1:5], b[None, :, 1:5]
    iw = np.clip(np.minimum(A[..., 2], B[..., 2]) - np.maximum(A[..., 0], B[..., 0]), 0, None)
    ih = np.clip(np.minimum(A[..., 3], B[..., 3]) - np.maximum(A[..., 1], B[..., 1]), 0, None)
    inter = iw * ih; ua = (A[..., 2] - A[..., 0]) * (A[..., 3] - A[..., 1]) + (B[..., 2] - B[..., 0]) * (B[..., 3] - B[..., 1]) - inter
    return float((inter / np.maximum(ua, 1e-9)).max(1).mean())

def prep_stmarc(cfg=V7):
    t0 = time.time(); vid = get("stmarc_video.avi")
    try: gt_raw = parse_polytrack(get("stmarc_annotations.zip"))
    except Exception as e: print("  주석 다운로드 실패:", repr(e)); gt_raw = None
    start = max(min(gt_raw) - 2, 0) if gt_raw else 0                 # 오프셋 확인용 여유 2프레임
    n = cfg["n_val"] if not gt_raw else min(cfg["n_val"], max(gt_raw) + 3 - start)
    ref = f"{UT}/stmarc_ref_{start}_{n}.mkv"
    if not os.path.exists(ref):
        r = subprocess.run(f'ffmpeg -y -loglevel error -i "{vid}" -vf "select=between(n\\,{start}\\,{start + n - 1})" -vsync 0 '
                           f'-c:v ffv1 -pix_fmt yuv420p -an "{ref}"', shell=True, capture_output=True, text=True)
        if r.returncode: raise RuntimeError(r.stderr[-2000:])
    for k in cfg["eval_kbps"]: encode_cctv(ref, k, f"{UT}/stmarc_{start}_{n}_{k}k.mp4")
    R_ = L.read_yuv(ref, n); n = len(R_[0]); ids = list(range(0, n, cfg["eval_step"]))
    # 의사 정답 (원본 프레임, conf 0.25, CCTV 클래스)
    pg = []
    for j in range(0, len(ids), 8):
        b = [L.frame_tensors(*(x[i] for x in R_))[0] for i in ids[j:j + 8]]
        for dt in L.detect(m, torch.stack(b).to(DEV), conf=0.25):
            dt = dt[np.isin(dt[:, 5], L.CCTV)]
            pg.append(np.column_stack([dt[:, 5], dt[:, 0], dt[:, 1], dt[:, 0] + dt[:, 2], dt[:, 1] + dt[:, 3]]) if len(dt) else np.zeros((0, 5)))
    gr = None
    if gt_raw:                                                        # 주석 프레임 번호 = 디코딩 순서 + off 중 IoU 최대
        sc = {}
        for off in range(-2, 3):
            v = [iou_match(np.array(gt_raw.get(start + i + off, []), float).reshape(-1, 5), p) for i, p in zip(ids, pg)]
            v = [x for x in v if x is not None]; sc[off] = round(float(np.mean(v)), 4) if v else 0.0
        off = max(sc, key=lambda o: (sc[o], -abs(o)))                  # 동률이면 0에 가까운 쪽
        print(f"  주석-의사정답 평균 최대 IoU (오프셋별): {sc} -> 사용 {off}")
        gr = [np.array(gt_raw.get(start + i + off, []), float).reshape(-1, 5) for i in ids]
        lo, hi = min(gt_raw), max(gt_raw)                               # 주석 구간 안 프레임만 평가 (객체 0개 프레임 포함)
        keep = [j for j, i in enumerate(ids) if lo <= start + i + off <= hi]
        print(f"  주석 구간 평가 프레임 {len(keep)}장 중 주석 0개 프레임 {sum(len(gr[j]) == 0 for j in keep)}장")
    else: keep = list(range(len(ids)))
    ids = [ids[j] for j in keep]; pg = [pg[j] for j in keep]; gr = [gr[j] for j in keep] if gr else None
    sub = lambda T: tuple(x[ids] for x in T)                         # 평가 프레임만 유지 (인코딩은 연속 프레임으로 끝난 뒤)
    d = {"ref": sub(R_), "comp": {k: sub(L.read_yuv(f"{UT}/stmarc_{start}_{n}_{k}k.mp4", n)) for k in cfg["eval_kbps"]},
         "gt": pg, "gt_real": gr, "frames": [start + i for i in ids]}
    d["comp"]["ref"] = d["ref"]                                        # 원본 그대로 평가(상한 참고, 보정 없음에서만)
    del R_
    a = np.concatenate(pg); sm = ((a[:, 3] - a[:, 1]) * (a[:, 4] - a[:, 2]) < L.SMALL).sum()
    print(f"  St-Marc 디코딩 {start}~{start + n - 1}, 평가 {len(ids)}장 (4프레임 간격), 의사 정답 {len(a)}개 (작은 객체 {sm}), {time.time() - t0:.0f}s")
    if gr:
        r = np.concatenate(gr); print(f"  실제 주석 {len(r)}개 (작은 객체 {((r[:, 3] - r[:, 1]) * (r[:, 4] - r[:, 2]) < L.SMALL).sum()})")
    DATA["StMarc"] = d
    import pickle                                                      # 부트스트랩용 정답 저장 (평가 프레임 순서 = preds의 frame 열)
    pickle.dump({"frames": d["frames"], "pseudo": pg, "real": gr}, open(f"{OUT}/stmarc_gt.pkl", "wb"))

def ap_rows(gts, dets, ids, classes, agnostic=False):
    """agnostic: CCTV 클래스 예측만 남기고 클래스 무시 NMS(IoU 0.7) 후 클래스 1로 통일 (같은 물체에 car·truck 박스가 둘 다 남는 경우 제거)"""
    import torchvision
    if agnostic:
        gts = [g.copy() for g in gts]; out = []
        for g in gts: g[:, 0] = 1
        for d in dets:
            d = d[np.isin(d[:, 5], L.CCTV)].copy()
            if len(d):
                b = torch.tensor(np.column_stack([d[:, 0], d[:, 1], d[:, 0] + d[:, 2], d[:, 1] + d[:, 3]]), dtype=torch.float32)
                d = d[torchvision.ops.nms(b, torch.tensor(d[:, 4], dtype=torch.float32), 0.7).numpy()]
            d[:, 5] = 1; out.append(d)
        dets = out
    G = L.coco_gt(gts); S = set(ids); G["images"] = [im for im in G["images"] if im["id"] in S]
    G["annotations"] = [a for a in G["annotations"] if a["image_id"] in S]
    full = [np.zeros((0, 6))] * len(gts)
    for i, d in zip(ids, dets): full[i] = d
    return L.coco_ap(G, L.to_coco([full[i] for i in ids], ids), [1] if agnostic else classes)

# ---------------- 손실 ----------------
def r_loss(f, fr, fc, mask=None):
    """샘플별 ‖(f−fr)·m‖² / ‖(fc−fr)·m‖² 평균 (= 1 − R). 마스크 영역이 빈 샘플 제외"""
    if mask is not None and mask.shape[-2:] != f.shape[-2:]: mask = F.interpolate(mask, size=f.shape[-2:], mode="nearest")
    mm = 1 if mask is None else mask
    num = (((f - fr) * mm) ** 2).flatten(1).sum(1); den = (((fc - fr) * mm) ** 2).flatten(1).sum(1)
    ok = den > 1e-8 * (fr ** 2).flatten(1).sum(1).clamp_min(1e-12)
    if mask is not None: ok = ok & (mask.flatten(1).sum(1) > 0)
    return (num[ok] / den[ok]).mean() if ok.any() else f.sum() * 0

def train7(spec, cfg=V7):
    cfg = {**cfg, **spec}; torch.manual_seed(cfg["seed"]); np.random.seed(cfg["seed"]); rng = np.random.default_rng(cfg["seed"])
    t0 = time.time(); ensure(TRAIN_ALL, cfg["kbps"]); print(f"  {spec['name']} 학습 시작 (데이터 로딩 {time.time() - t0:.0f}s)", flush=True)
    mod = make_mod(spec).to(DEV); frames = [(s, i) for s in TRAIN_ALL for i in range(len(DATA[s]["gt"]))]
    opt = torch.optim.AdamW(mod.parameters(), cfg["lr"], weight_decay=cfg["wd"]); scaler = torch.amp.GradScaler(enabled=AMP)
    T, W = cfg["steps"], cfg["warm"]
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda t: (t + 1) / W if t < W else 0.5 * (1 + math.cos(math.pi * (t - W) / max(1, T - W))))
    hist, agg, t0 = [], {}, time.time()
    for st in range(T):
        items = [(*frames[j], int(rng.choice(cfg["kbps"]))) for j in rng.choice(len(frames), cfg["batch"], replace=False)]
        rc, _, rr, _, gts = make_batch(items, cfg["flip"]); rc, rr = rc.to(DEV), rr.to(DEV)
        kb = torch.tensor([it[2] for it in items], device=DEV)
        mod.train(); m.eval()
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
            with torch.no_grad():
                fr = L.neck_feats(m, rr); fc = L.neck_feats(m, rc) if cfg["loss"] == "R" else None
            f = feats6(rc, mod, kb)
        f, fr = [x.float() for x in f], [x.float() for x in fr]
        ms = [x.to(DEV) for x in L.small_masks(gts, [(92, 160), (46, 80)])]
        if cfg["loss"] == "R":
            fc = [x.float() for x in fc]
            lg = sum(r_loss(f[q], fr[q], fc[q]) for q in range(3)); ls = r_loss(f[0], fr[0], fc[0], ms[0]) + r_loss(f[1], fr[1], fc[1], ms[1])
        else:
            lg = sum(L.rel_err(a, b)[0] for a, b in zip(f, fr)); ls = L.rel_err(f[0], fr[0], ms[0])[0] + L.rel_err(f[1], fr[1], ms[1])[0]
        loss = lg + cfg["lam_small"] * ls
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(mod.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step()
        for k, v in {"L_glob": lg, "L_small": ls}.items(): agg[k] = agg.get(k, 0) + v.item()
        if (st + 1) % 250 == 0:
            r = {k: round(v / 250, 4) for k, v in agg.items()}; r.update(step=st + 1, min=round((time.time() - t0) / 60, 2))
            hist.append(r); agg = {}; print("   ", r, flush=True)
    return mod, hist

@torch.no_grad()
def eval7(mod, tag, cfg=V7, kbs=None):
    d = DATA["StMarc"]; ids = list(range(len(d["gt"]))); rows = []
    kbs = kbs or cfg["eval_kbps"]; assert mod is None or "ref" not in kbs
    if mod is not None: mod.eval()
    m.eval()
    for k in kbs:
        num, den = np.zeros(4), np.zeros(4); dets = []
        for j in range(0, len(ids), 8):
            items = [("StMarc", i, k) for i in ids[j:j + 8]]; rc, _, rr, _, gts = make_batch(items, flip=False)
            rc, rr = rc.to(DEV), rr.to(DEV); kb = torch.full((len(items),), 0 if k == "ref" else k, device=DEV)
            fc = L.neck_feats(m, rc); fr = L.neck_feats(m, rr); f = fc if mod is None else feats6(rc, mod, kb)
            ms = L.small_masks(gts, [(92, 160)])[0].to(DEV)
            for q in range(3):
                num[q] += ((f[q] - fr[q]) ** 2).sum().item(); den[q] += ((fc[q] - fr[q]) ** 2).sum().item()
            num[3] += (((f[0] - fr[0]) * ms) ** 2).sum().item(); den[3] += (((fc[0] - fr[0]) * ms) ** 2).sum().item()
            dets += L.detect_from_feats(m, f)
        np.save(f"{OUT}/{tag}_preds_StMarc_{k}.npy",
                np.concatenate([np.column_stack([np.full(len(x), i), x]) for i, x in enumerate(dets)] or [np.zeros((0, 7))]).astype(np.float32))
        R = 1 - num / np.maximum(den, 1e-12)
        row = {"kbps": k, "R_P3": R[0], "R_P4": R[1], "R_P5": R[2], "R_P3small": R[3]}
        ap = ap_rows(d["gt"], dets, ids, L.CCTV)
        if ap: row.update({f"{kk}_pseudo": ap[kk] for kk in ("AP", "APs", "APm", "APl")})
        if d["gt_real"] is not None:
            ap = ap_rows(d["gt_real"], dets, ids, None, agnostic=True)
            if ap: row.update({f"{kk}_gt": ap[kk] for kk in ("AP", "AP50", "APs", "APm", "APl")})
        rows.append({kk: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for kk, v in row.items()})
    return rows

# ==== v7 실행 ====
RUNS7 = globals().get("RUNS7", [
    dict(name="B_small", kind="B"),                                                   # v6 최고 (대조군)
    dict(name="B_small_R", kind="B", loss="R"),
    dict(name="B_small_cond_R", kind="B", cond=True, loss="R"),
    dict(name="B_small_cond_R_4k", kind="B", cond=True, loss="R", kbps=[4000, 2000, 1000, 500, 250]),
    dict(name="B_small_cond_R_s1", kind="B", cond=True, loss="R", seed=1),           # 시드 차이 확인
])
RES7 = []; t_all = time.time()
print("##### St-Marc 준비 #####", flush=True); prep_stmarc()
for r in eval7(None, "none", kbs=["ref"] + V7["eval_kbps"]): RES7.append({"run": "none", **r}); print("  none", r, flush=True)
pd.DataFrame(RES7).to_csv(f"{OUT}/v7_results_{STAMP}.csv", index=False)
for spec in RUNS7:
    t0 = time.time(); mod, hist = train7(spec); n_p, g = cost(mod)
    torch.save({"spec": spec, "state": mod.state_dict(), "hist": hist, "V7": V7}, f"{OUT}/v7_{spec['name']}.pt")
    for r in eval7(mod, spec["name"]): RES7.append({"run": spec["name"], "params": n_p, "GFLOPs": g, **r}); print(" ", spec["name"], r, flush=True)
    print(f"  {spec['name']} 끝 {(time.time() - t0) / 60:.1f}분 (누적 {(time.time() - t_all) / 60:.1f}분)", flush=True)
    pd.DataFrame(RES7).to_csv(f"{OUT}/v7_results_{STAMP}.csv", index=False)
    del mod; torch.cuda.empty_cache()
print(pd.DataFrame(RES7).to_string())


In [ ]:
%%writefile /kaggle/working/fcb_v82.py
# FCB v8-2: 학습 데이터 양·종류에 따른 B_small_R 일반화 (모듈·손실 고정, 데이터만 변경)
# 검증 장면 2개 (Urban Tracker, 고정 CCTV, 1280×720): St-Marc(주석 1000~1999), René-Lévesque(주석 구간, 주석 영역 마스크 있음)
# 학습 데이터: UVG 학습 시퀀스(2/4/6개) / + Rouen·Sherbrooke(다른 교차로 CCTV, 라벨 없이 원본 feature 맞추기) /
#             + St-Marc 0~997프레임(검증과 같은 카메라, 다른 시간대: "설치 카메라 적응" 시나리오, 일반화 결과 아님)
# Urban Tracker 영상·주석은 노트북 안에서 원 사이트에서 다운로드 (재업로드 없음). 평가셋 5개 사용 안 함
exec(open("/kaggle/working/fcb_v7.py").read().split("# ==== v7 실행 ====")[0])
import cv2
OUT = "/kaggle/working/fcb_v82_runs"; os.makedirs(OUT, exist_ok=True)
UTB = "https://urbantracker.jpjodoin.com/dataset"
SCENES = {   # vf: 1280×720로 맞추는 필터 (비율 유지, 남는 곳은 회색 114 패딩)
    "stmarc": dict(video="stmarc/stmarc_video.avi", ann="stmarc/stmarc_annotations.zip", mask=None, vf=None),
    "rene": dict(video="rene/rene_video.mov", ann="rene/rene_annotations.zip", mask="rene/rene_mask.png", vf=None),
    "rouen": dict(video="rouen/rouen_video.avi", vf="scale=1280:720:flags=bicubic"),
    "sherbrooke": dict(video="sherbrooke/sherbrooke_video.avi", vf="scale=960:720:flags=bicubic,pad=1280:720:160:0:color=0x727272"),
}
V82 = dict(V7, loss="R", lam_small=4.0, steps=1500, kbps=[2000, 1000, 500, 250], eval_kbps=[1000, 500, 250],
           eval_step=4, n_val=1000, train_step=2, n_train=600)

def dl(rel):
    p = f"{UT}/{rel.replace('/', '_')}"
    if not os.path.exists(p): urllib.request.urlretrieve(f"{UTB}/{rel}", p)
    return p

def parse_ann(zp, scene):
    """Polytrack zip 안 sqlite 중 bounding_boxes 행이 가장 많은 파일(= 전체 도로 이용자) 사용 -> {frame: [[1,x1,y1,x2,y2],...]}"""
    try:
        d = f"{UT}/ann_{scene}"; zipfile.ZipFile(zp).extractall(d)
        files = [os.path.join(r, f) for r, _, fs in os.walk(d) for f in fs]
        print(f"  [{scene}] 주석 파일:", [os.path.relpath(x, d) for x in files])
        for x in files:
            if x.lower().endswith("readme.txt"): print("  ReadMe:", " ".join(open(x, errors="ignore").read().split())[:400])
        best, nb = None, -1
        for x in files:
            if not x.lower().endswith((".sqlite", ".db", ".sqlite3")): continue
            try:
                c = sqlite3.connect(x).cursor().execute("select count(*) from bounding_boxes").fetchone()[0]
                print(f"    {os.path.basename(x)}: bounding_boxes {c}행")
                if c > nb: best, nb = x, c
            except Exception as e: print(f"    {os.path.basename(x)}: {e}")
        if best is None: return None
        cur = sqlite3.connect(best).cursor(); print("  사용:", os.path.basename(best))
        for q, lab in [("select * from objects_type", "objects_type"),
                       ("select road_user_type, count(*) from objects group by road_user_type", "road_user_type 분포")]:
            try: print(f"  {lab}:", cur.execute(q).fetchall())
            except Exception as e: print(f"  {lab} 없음:", e)
        g = {}
        for f, x1, y1, x2, y2 in cur.execute("select frame_number, x_top_left, y_top_left, x_bottom_right, y_bottom_right from bounding_boxes"):
            g.setdefault(int(f), []).append([1, min(x1, x2), min(y1, y2), max(x1, x2), max(y1, y2)])
        fs = sorted(g); print(f"  주석 프레임 {fs[0]}~{fs[-1]} ({len(fs)}개), 박스 {sum(len(v) for v in g.values())}개")
        return g
    except Exception as e:
        print(f"  [{scene}] 주석 실패:", repr(e)); return None

def decode(scene, start, n):
    vid = dl(SCENES[scene]["video"]); vf = f"select=between(n\\,{start}\\,{start + n - 1})"
    if SCENES[scene]["vf"]: vf += "," + SCENES[scene]["vf"]
    ref = f"{UT}/{scene}_ref_{start}_{n}.mkv"
    if not os.path.exists(ref):
        r = subprocess.run(f'ffmpeg -y -loglevel error -i "{vid}" -vf "{vf}" -vsync 0 -c:v ffv1 -pix_fmt yuv420p -an "{ref}"',
                           shell=True, capture_output=True, text=True)
        if r.returncode: raise RuntimeError(r.stderr[-2000:])
    return ref

@torch.no_grad()
def pseudo_gt(Rf, ids):
    """원본 프레임에 고정 탐지기 (conf 0.25, CCTV 클래스) -> [cat, x1, y1, x2, y2]"""
    out = []
    for j in range(0, len(ids), 8):
        b = [L.frame_tensors(*(x[i] for x in Rf))[0] for i in ids[j:j + 8]]
        for dt in L.detect(m, torch.stack(b).to(DEV), conf=0.25):
            dt = dt[np.isin(dt[:, 5], L.CCTV)]
            out.append(np.column_stack([dt[:, 5], dt[:, 0], dt[:, 1], dt[:, 0] + dt[:, 2], dt[:, 1] + dt[:, 3]]) if len(dt) else np.zeros((0, 5)))
    return out

def prep_val(scene, cfg=V82):
    """주석 구간을 연속 디코딩 -> 압축 -> 주석 구간 안 프레임만 eval_step 간격으로 유지. DATA[f'{scene}_val']"""
    t0 = time.time(); g = parse_ann(dl(SCENES[scene]["ann"]), scene)
    if not g: raise RuntimeError(f"{scene} 주석 없음")
    nv = int(subprocess.run(f'ffprobe -v error -select_streams v:0 -count_packets -show_entries stream=nb_read_packets -of csv=p=0 "{dl(SCENES[scene]['video'])}"',
                            shell=True, capture_output=True, text=True).stdout.strip().split()[0] or 0)
    lo, hi = min(g), max(g); base = 0 if hi < nv else lo        # 주석 번호가 영상 길이를 넘으면 영상 첫 프레임 = 주석 첫 프레임으로 봄
    print(f"  [{scene}] 영상 {nv}프레임, 주석 {lo}~{hi} -> 영상 프레임 = 주석 번호 - {base}", flush=True)
    g = {f - base: v for f, v in g.items()}; lo, hi = lo - base, hi - base
    start = max(lo - 2, 0); n = min(cfg["n_val"] + 4, hi + 3 - start)
    ref = decode(scene, start, n)
    for k in cfg["eval_kbps"]: encode_cctv(ref, k, f"{UT}/{scene}_{start}_{n}_{k}k.mp4")
    n0 = n; Rf = L.read_yuv(ref, n); n = len(Rf[0]); ids = list(range(0, n, cfg["eval_step"])); pg = pseudo_gt(Rf, ids)
    sc = {}
    for off in range(-2, 3):
        v = [iou_match(np.array(g.get(start + i + off, []), float).reshape(-1, 5), p) for i, p in zip(ids, pg)]
        v = [x for x in v if x is not None]; sc[off] = round(float(np.mean(v)), 4) if v else 0.0
    off = max(sc, key=lambda o: (sc[o], -abs(o))); print(f"  [{scene}] 주석-의사정답 평균 최대 IoU (오프셋별): {sc} -> 사용 {off}")
    keep = [j for j, i in enumerate(ids) if lo <= start + i + off <= hi]
    ids = [ids[j] for j in keep]; pg = [pg[j] for j in keep]
    gr = [np.array(g.get(start + i + off, []), float).reshape(-1, 5) for i in ids]
    sub = lambda T: tuple(x[ids] for x in T)
    d = {"ref": sub(Rf), "comp": {k: sub(L.read_yuv(f"{UT}/{scene}_{start}_{n0}_{k}k.mp4", n)) for k in cfg["eval_kbps"]},
         "gt": pg, "gt_real": gr, "frames": [start + i for i in ids], "mask": None}
    d["comp"]["ref"] = d["ref"]; del Rf
    if SCENES[scene].get("mask"):
        mk = cv2.imread(dl(SCENES[scene]["mask"]), cv2.IMREAD_GRAYSCALE)
        if mk is not None:
            if mk.shape != (720, 1280): mk = cv2.resize(mk, (1280, 720), interpolation=cv2.INTER_NEAREST)
            d["mask"] = mk > 127; print(f"  [{scene}] 주석 영역 마스크 {d['mask'].mean() * 100:.1f}% (영역 밖 중심 예측은 AP_gt에서 제외)")
    a = np.concatenate(pg); r = np.concatenate(gr)
    print(f"  [{scene}] 프레임 {start}~{start + n - 1} 디코딩, 평가 {len(ids)}장, 의사 정답 {len(a)}개 (작은 {((a[:, 3] - a[:, 1]) * (a[:, 4] - a[:, 2]) < L.SMALL).sum()}), "
          f"실제 주석 {len(r)}개 (작은 {((r[:, 3] - r[:, 1]) * (r[:, 4] - r[:, 2]) < L.SMALL).sum()}), 주석 0개 프레임 {sum(len(x) == 0 for x in gr)}, {time.time() - t0:.0f}s", flush=True)
    DATA[f"{scene}_val"] = d
    import pickle
    pickle.dump({"frames": d["frames"], "pseudo": pg, "real": gr, "mask": d["mask"]}, open(f"{OUT}/{scene}_gt.pkl", "wb"))

def prep_train(name, scene, start, n, cfg=V82):
    """학습용 장면: 연속 디코딩 -> 학습 비트레이트로 압축 -> train_step 간격 프레임만 유지. 작은 객체 마스크는 의사 정답으로"""
    t0 = time.time(); ref = decode(scene, start, n)
    for k in cfg["kbps"]: encode_cctv(ref, k, f"{UT}/{scene}_{start}_{n}_{k}k.mp4")
    n0 = n; Rf = L.read_yuv(ref, n); n = len(Rf[0]); ids = list(range(0, n, cfg["train_step"]))
    sub = lambda T: tuple(x[ids] for x in T)
    d = {"ref": sub(Rf), "comp": {k: sub(L.read_yuv(f"{UT}/{scene}_{start}_{n0}_{k}k.mp4", n)) for k in cfg["kbps"]}}
    d["gt"] = pseudo_gt(Rf, ids); del Rf; DATA[name] = d
    a = np.concatenate(d["gt"]); print(f"  [{name}] 학습 프레임 {len(ids)}장 (원본 {start}~{start + n - 1}, {cfg['train_step']}프레임 간격), 의사 정답 {len(a)}개 (작은 {((a[:, 3] - a[:, 1]) * (a[:, 4] - a[:, 2]) < L.SMALL).sum()}), {time.time() - t0:.0f}s", flush=True)

def train8(spec, cfg=V82):
    cfg = {**cfg, **spec}; torch.manual_seed(cfg["seed"]); np.random.seed(cfg["seed"]); rng = np.random.default_rng(cfg["seed"])
    seqs = spec["data"]; t0 = time.time(); ensure([s for s in seqs if s in TRAIN_ALL], cfg["kbps"])
    frames = [(s, i) for s in seqs for i in range(len(DATA[s]["gt"]))]
    print(f"  {spec['name']} 학습 시작: {seqs} 프레임 {len(frames)}장 (로딩 {time.time() - t0:.0f}s)", flush=True)
    mod = make_mod(spec).to(DEV)
    opt = torch.optim.AdamW(mod.parameters(), cfg["lr"], weight_decay=cfg["wd"]); scaler = torch.amp.GradScaler(enabled=AMP)
    T, W = cfg["steps"], cfg["warm"]
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda t: (t + 1) / W if t < W else 0.5 * (1 + math.cos(math.pi * (t - W) / max(1, T - W))))
    hist, agg, t0 = [], {}, time.time()
    for st in range(T):
        items = [(*frames[j], int(rng.choice(cfg["kbps"]))) for j in rng.choice(len(frames), cfg["batch"], replace=False)]
        rc, _, rr, _, gts = make_batch(items, cfg["flip"]); rc, rr = rc.to(DEV), rr.to(DEV)
        kb = torch.tensor([it[2] for it in items], device=DEV); mod.train(); m.eval()
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
            with torch.no_grad(): fr = L.neck_feats(m, rr); fc = L.neck_feats(m, rc)
            f = feats6(rc, mod, kb)
        f, fr, fc = [x.float() for x in f], [x.float() for x in fr], [x.float() for x in fc]
        ms = [x.to(DEV) for x in L.small_masks(gts, [(92, 160), (46, 80)])]
        lg = sum(r_loss(f[q], fr[q], fc[q]) for q in range(3)); ls = r_loss(f[0], fr[0], fc[0], ms[0]) + r_loss(f[1], fr[1], fc[1], ms[1])
        loss = lg + cfg["lam_small"] * ls
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(mod.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step()
        for k, v in {"L_glob": lg, "L_small": ls}.items(): agg[k] = agg.get(k, 0) + v.item()
        if (st + 1) % 250 == 0:
            r = {k: round(v / 250, 4) for k, v in agg.items()}; r.update(step=st + 1, min=round((time.time() - t0) / 60, 2))
            hist.append(r); agg = {}; print("   ", r, flush=True)
    return mod, hist

def in_mask(d, mask):
    """René(마스크 있는 장면): 보행자 주석 없음 -> person 예측 제외 + 주석 영역 밖 제외"""
    if mask is None or len(d) == 0: return d
    d = d[d[:, 5] != 1]
    if len(d) == 0: return d
    cx = np.clip((d[:, 0] + d[:, 2] / 2).astype(int), 0, 1279); cy = np.clip((d[:, 1] + d[:, 3] / 2).astype(int), 0, 719)
    return d[mask[cy, cx]]

@torch.no_grad()
def eval8(mod, tag, scene, cfg=V82, kbs=None):
    key = f"{scene}_val"; d = DATA[key]; ids = list(range(len(d["gt"]))); rows = []
    kbs = kbs or cfg["eval_kbps"]; assert mod is None or "ref" not in kbs
    if mod is not None: mod.eval()
    m.eval()
    for k in kbs:
        num, den = np.zeros(4), np.zeros(4); dets = []
        for j in range(0, len(ids), 8):
            items = [(key, i, k) for i in ids[j:j + 8]]; rc, _, rr, _, gts = make_batch(items, flip=False)
            rc, rr = rc.to(DEV), rr.to(DEV); kb = torch.full((len(items),), 0 if k == "ref" else k, device=DEV)
            fc = L.neck_feats(m, rc); fr = L.neck_feats(m, rr); f = fc if mod is None else feats6(rc, mod, kb)
            ms = L.small_masks(gts, [(92, 160)])[0].to(DEV)
            for q in range(3):
                num[q] += ((f[q] - fr[q]) ** 2).sum().item(); den[q] += ((fc[q] - fr[q]) ** 2).sum().item()
            num[3] += (((f[0] - fr[0]) * ms) ** 2).sum().item(); den[3] += (((fc[0] - fr[0]) * ms) ** 2).sum().item()
            dets += L.detect_from_feats(m, f)
        np.save(f"{OUT}/{tag}_preds_{scene}_{k}.npy",
                np.concatenate([np.column_stack([np.full(len(x), i), x]) for i, x in enumerate(dets)] or [np.zeros((0, 7))]).astype(np.float32))
        R = 1 - num / np.maximum(den, 1e-12)
        row = {"scene": scene, "kbps": k, "R_P3": R[0], "R_P4": R[1], "R_P5": R[2], "R_P3small": R[3]}
        ap = ap_rows(d["gt_real"], [in_mask(x, d["mask"]) for x in dets], ids, None, agnostic=True)
        if ap: row.update({f"{kk}_gt": ap[kk] for kk in ("AP", "AP50", "APs", "APm", "APl")})
        ap = ap_rows(d["gt"], dets, ids, L.CCTV)
        if ap: row.update({f"{kk}_pseudo": ap[kk] for kk in ("AP", "APs")})
        rows.append({kk: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for kk, v in row.items()})
    return rows

# ==== v8-2 실행 ====
U2 = ["HighwayDrive", "TrafficLights"]; U4 = U2 + ["WinterDrive", "OfficeWalk"]; U6 = list(TRAIN_ALL)
RUNS82 = globals().get("RUNS82", [
    dict(name="U2", kind="B", data=U2),
    dict(name="U4", kind="B", data=U4),
    dict(name="U6", kind="B", data=U6),                                              # = v7 B_small_R 설정
    dict(name="U6_UT", kind="B", data=U6 + ["rouen_tr", "sherbrooke_tr"]),
    dict(name="U6_UT_SM", kind="B", data=U6 + ["rouen_tr", "sherbrooke_tr", "stmarc_tr"]),   # 같은 카메라 적응 시나리오
])
VAL = globals().get("VAL", ["stmarc", "rene"])
RES82 = []; t_all = time.time()
print("##### 검증 장면 준비 #####", flush=True)
for sc_ in VAL: prep_val(sc_)
for sc_ in VAL:
    for r in eval8(None, "none", sc_, kbs=["ref"] + V82["eval_kbps"]): RES82.append({"run": "none", **r}); print("  none", r, flush=True)
pd.DataFrame(RES82).to_csv(f"{OUT}/v82_results_{STAMP}.csv", index=False)
for spec in RUNS82:
    need = [s for s in spec["data"] if s.endswith("_tr") and s not in DATA] if not os.path.exists(f"{OUT}/v82_{spec['name']}.pt") else []
    for s in need:
        if s == "stmarc_tr": prep_train(s, "stmarc", 0, 998)                       # 검증(998~)과 겹치지 않는 앞부분
        else: prep_train(s, s[:-3], 0, V82["n_train"])
    t0 = time.time(); ck = f"{OUT}/v82_{spec['name']}.pt"
    if os.path.exists(ck):                                                     # 세션이 끊겼다 다시 돌릴 때: 학습된 run은 평가만
        mod = make_mod(spec).to(DEV); mod.load_state_dict(torch.load(ck, map_location=DEV)["state"]); print(f"  {spec['name']} 저장된 가중치 사용", flush=True)
    else:
        mod, hist = train8(spec)
        torch.save({"spec": spec, "state": mod.state_dict(), "hist": hist, "V82": V82}, ck)
    n_p, g = cost(mod)
    for sc_ in VAL:
        for r in eval8(mod, spec["name"], sc_): RES82.append({"run": spec["name"], "params": n_p, "GFLOPs": g, **r}); print(" ", spec["name"], r, flush=True)
    print(f"  {spec['name']} 끝 {(time.time() - t0) / 60:.1f}분 (누적 {(time.time() - t_all) / 60:.1f}분)", flush=True)
    pd.DataFrame(RES82).to_csv(f"{OUT}/v82_results_{STAMP}.csv", index=False)
    del mod; torch.cuda.empty_cache()
D82 = pd.DataFrame(RES82)
for sc_ in VAL:
    for col in ["AP_gt", "APs_gt", "R_P3", "R_P3small"]:
        print(f"== {sc_} {col}"); print(D82[D82.scene == sc_].pivot(index="run", columns="kbps", values=col).to_string())


In [ ]:
%%writefile /kaggle/working/fcb_v83.py
# FCB v8-3: U6_UT 개선 시도 (데이터 확대 / 학습 길이·용량), 시드 차이 확인
# 기준: v8-2 U6_UT (B 어댑터 24/48ch, 손실 R + 작은 객체 4, UVG 6개 + Rouen·Sherbrooke 각 0~599 2프레임 간격, 1500 step)
# 모든 run을 같은 세션·같은 압축 파일로 비교 (세션이 바뀌면 보정 없음 기준값도 흔들림: v8-2 두 세션 St-Marc APs 1000 16.24 vs 14.31)
# 검증: St-Marc(실제 주석 AP·APs, 주 지표), René(R만 참고: AP_gt가 원본보다 높게 나와 지표로 쓰지 않음)
# 평가셋 5개 사용 안 함. 같은 카메라(St-Marc 0~997) 데이터 사용 안 함
exec(open("/kaggle/working/fcb_v82.py").read().split("# ==== v8-2 실행 ====")[0])
OUT = "/kaggle/working/fcb_v83_runs"; os.makedirs(OUT, exist_ok=True)
V83 = dict(V82, eval_kbps=[2000, 1000, 500, 250])

def ram():
    try:
        import psutil; v = psutil.virtual_memory(); return f"RAM {v.used / 2**30:.1f}/{v.total / 2**30:.1f}GB"
    except Exception: return ""

def read_yuv_sub(video, n, step):
    """연속 디코딩하면서 step 간격 프레임만 미리 잡은 배열에 저장 (read_yuv의 2배 메모리 피크 없음)"""
    Wd, Hd = L.W, L.H; mm_ = (n + step - 1) // step
    Y = np.empty((mm_, Hd, Wd), np.uint8); U = np.empty((mm_, Hd // 2, Wd // 2), np.uint8); V = np.empty_like(U)
    p = subprocess.Popen(["ffmpeg", "-v", "error", "-i", video, "-f", "rawvideo", "-pix_fmt", "yuv420p", "-"], stdout=subprocess.PIPE)
    fb = Wd * Hd * 3 // 2; i = j = 0
    while i < n:
        b = p.stdout.read(fb)
        if len(b) < fb: break
        if i % step == 0:
            a = np.frombuffer(b, np.uint8); Y[j] = a[:Wd * Hd].reshape(Hd, Wd)
            U[j] = a[Wd * Hd:Wd * Hd * 5 // 4].reshape(Hd // 2, Wd // 2); V[j] = a[Wd * Hd * 5 // 4:].reshape(Hd // 2, Wd // 2); j += 1
        i += 1
    p.stdout.close(); p.wait()
    if j == 0: raise RuntimeError(f"{video}: 0프레임")
    return Y[:j], U[:j], V[:j]

def prep_train_x(name, scene, start, n, step, cfg=V83):
    """학습 장면: 연속 디코딩 -> 학습 비트레이트로 압축(연속 프레임) -> step 간격만 메모리에. 작은 객체 마스크는 의사 정답"""
    t0 = time.time(); ref = decode(scene, start, n)
    for k in cfg["kbps"]: encode_cctv(ref, k, f"{UT}/{scene}_{start}_{n}_{k}k.mp4")
    Rf = read_yuv_sub(ref, n, step)
    d = {"ref": Rf, "comp": {k: read_yuv_sub(f"{UT}/{scene}_{start}_{n}_{k}k.mp4", n, step) for k in cfg["kbps"]}}
    assert all(len(c[0]) == len(Rf[0]) for c in d["comp"].values()), "압축본 프레임 수가 원본과 다름"
    d["gt"] = pseudo_gt(Rf, list(range(len(Rf[0])))); DATA[name] = d
    a = np.concatenate(d["gt"])
    print(f"  [{name}] 원본 {start}~{start + n - 1}, {step}프레임 간격 {len(Rf[0])}장, 의사 정답 {len(a)}개 (작은 {((a[:, 3] - a[:, 1]) * (a[:, 4] - a[:, 2]) < L.SMALL).sum()}), {time.time() - t0:.0f}s, {ram()}", flush=True)

def view(name, src, sl):
    """src 장면 데이터의 일부를 복사 없이 참조 (numpy 기본 슬라이싱 = view)"""
    s_ = DATA[src]; DATA[name] = {"ref": tuple(x[sl] for x in s_["ref"]), "comp": {k: tuple(x[sl] for x in v) for k, v in s_["comp"].items()},
                                  "gt": s_["gt"][sl]}
    print(f"  [{name}] = {src}[{sl.start}:{sl.stop}:{sl.step}] {len(DATA[name]['gt'])}장 (복사 없음)", flush=True)

def make_mod(spec):                          # v6 make_mod + 어댑터 채널 지정
    if spec["kind"] == "B": return ModB6(cond=spec.get("cond", False), m2=spec.get("m2", 24), m4=spec.get("m4", 48))
    return MODS[spec["kind"]]()

# ==== v8-3 실행 ====
U6 = list(TRAIN_ALL); UTD = U6 + ["rouen_tr", "sherbrooke_tr"]; UTX = U6 + ["rouenX", "sherbrookeX"]   # UT는 Urban Tracker 폴더 경로 변수라 다른 이름
RUNS83 = globals().get("RUNS83", [
    dict(name="UT_s0", kind="B", data=UTD, seed=0),                                   # = v8-2 U6_UT (같은 세션 기준점)
    dict(name="UT_s1", kind="B", data=UTD, seed=1),                                   # 시드 차이
    dict(name="UTx", kind="B", data=UTX, seed=0),                                    # CCTV 데이터 약 2배
    dict(name="UTx_lb", kind="B", data=UTX, seed=0, steps=3000, m2=32, m4=64),        # + 학습 2배, 채널 32/64
])
VAL = globals().get("VAL", ["stmarc", "rene"]); RES83 = []; t_all = time.time()
print("##### 검증 장면 준비 #####", flush=True)
for sc_ in VAL: prep_val(sc_, V83)
print(" ", ram(), flush=True)
for sc_ in VAL:
    for r in eval8(None, "none", sc_, V83, kbs=["ref"] + V83["eval_kbps"]): RES83.append({"run": "none", **r}); print("  none", r, flush=True)
pd.DataFrame(RES83).to_csv(f"{OUT}/v83_results_{STAMP}.csv", index=False)
print("##### 학습 장면 준비 #####", flush=True)
ensure(U6, V83["kbps"]); print("  UVG 6개", ram(), flush=True)
prep_train_x("rouenX", "rouen", 0, V83.get("n_rouen", 630), 1)                       # Rouen 전체 630프레임, 1프레임 간격
prep_train_x("sherbrookeX", "sherbrooke", 0, V83.get("n_sher", 1200), 2)             # Sherbrooke 0~1199, 2프레임 간격 (600장)
view("rouen_tr", "rouenX", slice(0, 600, 2))                                         # = v8-2 rouen_tr (0~599, 2프레임 간격)
view("sherbrooke_tr", "sherbrookeX", slice(0, 300, 1))                               # = v8-2 sherbrooke_tr (0~598 짝수 프레임)
for spec in RUNS83:
    t0 = time.time(); ck = f"{OUT}/v83_{spec['name']}.pt"
    if os.path.exists(ck):
        mod = make_mod(spec).to(DEV); mod.load_state_dict(torch.load(ck, map_location=DEV)["state"]); print(f"  {spec['name']} 저장된 가중치 사용", flush=True)
    else:
        mod, hist = train8(spec, V83)
        torch.save({"spec": spec, "state": mod.state_dict(), "hist": hist, "V83": V83}, ck)
    n_p, g = cost(mod)
    for sc_ in VAL:
        for r in eval8(mod, spec["name"], sc_, V83): RES83.append({"run": spec["name"], "params": n_p, "GFLOPs": g, **r}); print(" ", spec["name"], r, flush=True)
    print(f"  {spec['name']} 끝 {(time.time() - t0) / 60:.1f}분 (누적 {(time.time() - t_all) / 60:.1f}분) {ram()}", flush=True)
    pd.DataFrame(RES83).to_csv(f"{OUT}/v83_results_{STAMP}.csv", index=False)
    del mod; torch.cuda.empty_cache()
D83 = pd.DataFrame(RES83)
for sc_ in VAL:
    for col in (["AP_gt", "APs_gt", "R_P3", "R_P3small"] if sc_ == "stmarc" else ["R_P3", "R_P3small"]):
        print(f"== {sc_} {col}"); print(D83[D83.scene == sc_].pivot(index="run", columns="kbps", values=col).to_string())


In [ ]:
import warnings, sys; warnings.filterwarnings('ignore')
sys.modules.pop('fcb_lib', None)
exec(open('/kaggle/working/fcb_v83.py').read())   # 검증 장면 -> 보정 없음 -> 학습 장면 -> 4개 학습·검증 -> 표

In [ ]:
!cd /kaggle/working && rm -f fcb_v83_runs.zip && zip -qr fcb_v83_runs.zip fcb_v83_runs && ls -lh fcb_v83_runs.zip